# Capítulo 15: Aprendizado Não Supervisionado

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 12 de James et al. (2023).

> Em muitas investigações físicas, estatísticas e biológicas, é desejável representar um sistema de pontos no plano, ou no espaço de três ou mais dimensões, pela reta ou pelo plano de "melhor ajuste".
>
> — Karl Pearson, em tradução livre (1901, p. 559)

Uma tabela pode não ter coluna nenhuma a prever: os 50 estados do `USArrests`, com as prisões por três crimes e a parcela da população urbana, ou as 64 amostras de câncer do `NCI60`, com a expressão de 6.830 genes. Sem uma resposta $Y$, não há erro de teste a medir. A seção 15.1 fixa os dois objetivos mais comuns do aprendizado não supervisionado: resumir as variáveis em poucas variáveis novas, que guardem o máximo da variação, e encontrar subgrupos de observações parecidas. No `NCI60`, a coluna `tipo`, com o tipo de câncer de cada amostra, fica fora dos métodos e serve só para comparar depois; um agrupamento que não coincide com ela não está, por isso, errado.

As componentes principais são as direções em que os dados mais variam. Na seção 15.2, com `homicidio` e `agressao` padronizadas, a reta de 45 graus é ao mesmo tempo a que mais espalha as projeções dos estados, com variância 1,802, e a que passa mais perto deles. Com as quatro colunas, a primeira componente dá às três colunas de crime cargas entre 0,54 e 0,58, quase o mesmo peso, e a segunda mede sobretudo a urbanização. O sinal de uma componente é arbitrário: o que se lê é o tamanho das cargas e quais têm o mesmo sinal. A seção 15.3 mede quanto cada componente carrega. As duas primeiras explicam 0,8675 da variância do `USArrests` padronizado, e a proporção acumulada é o complemento do erro relativo da aproximação, $1 - \text{RSS}/\text{TSS}$. Sem padronizar, `agressao`, que tem sozinha 0,956 da variância total, vira quase toda a primeira componente; ainda sem padronizar, com `agressao` contada a cada 100 habitantes em vez de 100.000, quem passa a dominar é `pop_urbana`, e padronizadas as duas unidades dão o mesmo resumo. No `NCI60`, a primeira componente explica 0,114 da variância, e o gráfico do cotovelo é bem menos nítido que o dos estados: quantas componentes guardar é uma decisão de quem analisa.

O *k*-means, da seção 15.4, divide as observações em $K$ grupos procurando a menor variação dentro deles, alternando médias e reatribuições, e nenhum passo aumenta o objetivo. Ele para num ótimo local: nos 150 pontos simulados, seis sorteios iniciais param em três ótimos locais diferentes, e por isso roda a partir de vários começos. O agrupamento hierárquico, da seção 15.5, não pede $K$: constrói uma árvore de fusões, o dendrograma, e o número de grupos sai de um corte feito depois. Duas folhas lado a lado no desenho não são, por isso, parecidas; o que diz quão parecidas elas são é a altura em que se juntam. A árvore depende da ligação e da medida de dissimilaridade. No `NCI60`, cortadas em 4 grupos, as ligações completa, média e simples deixam 40, 54 e 61 amostras no maior grupo, e, na simples, o maior grupo cresce encadeado, na maior parte das fusões uma amostra por vez. Com a ligação completa, as 6 amostras de leucemia ficam num grupo de 8, com as duas medições repetidas da linhagem K562, também de leucemia.

Cada decisão pequena pode mudar os grupos, e a seção 15.6 mede algumas. Sem padronizar, os três grupos dos estados são só três faixas de `agressao`. A inércia cai a cada grupo a mais, também em ruído sem grupo nenhum, que o *k*-means divide em três grupos quando se pedem três; por isso procurar a menor inércia não escolhe $K$. Duas sementes podem dar a mesma divisão com os rótulos trocados: nos estados, só 0,48 deles receberam o mesmo rótulo nas duas, e por isso agrupamentos se comparam por tabela cruzada ou pelo índice de Rand ajustado. No `NCI60`, dez sementes do *k*-means sobre os genes deram 9 divisões diferentes, e sobre os 5 primeiros escores da `PCA`, uma só. Em 20 subamostras de 40 estados, a divisão em 2 grupos se manteve em todas, e a em 3 mudou na maioria delas, em geral pouco, e em 5 ficou com índice abaixo de 0,5. O que se repete sob escolhas diferentes é o que vale relatar, e mesmo assim como hipótese a conferir em dados novos.

Ao final deste capítulo, você será capaz de:

- Explicar o que muda quando não há resposta $Y$, quais são os dois objetivos do aprendizado não supervisionado e por que comparar grupos com uma coluna externa não diz se eles estão certos
- Calcular as componentes principais com `make_pipeline`, `StandardScaler` e `PCA`, ler cargas e escores pelo que não depende do sinal e interpretar um *biplot*
- Medir a proporção da variância explicada, relacioná-la ao erro da aproximação de dimensão $M$, decidir se as colunas devem ser padronizadas e discutir quantas componentes guardar com o gráfico do cotovelo
- Descrever o algoritmo do *k*-means, relacionar o objetivo à inércia (`inertia_`) e justificar o uso de vários começos (`n_init`) e de uma semente (`random_state`)
- Construir, ler e cortar um dendrograma, escolher a ligação e a medida de dissimilaridade, euclidiana ou baseada em correlação, e obter os mesmos grupos com o `AgglomerativeClustering`
- Avaliar um agrupamento sem resposta: comparar divisões por tabela cruzada e pelo índice de Rand ajustado, não escolher $K$ pela inércia e conferir se os grupos resistem à troca de escala, de semente e de ligação e à perda de observações

## Seções

| Seção | Tópico |
|---|---|
| [15.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/01-o-desafio-do-nao-supervisionado.html) | O Desafio do Não Supervisionado |
| [15.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/02-componentes-principais.html) | Componentes Principais |
| [15.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/03-proporcao-da-variancia-explicada.html) | Proporção da Variância Explicada |
| [15.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/04-k-means.html) | k-Means |
| [15.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/05-clustering-hierarquico.html) | Clustering Hierárquico |
| [15.6](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap15/06-questoes-praticas-em-clustering.html) | Questões Práticas em Clustering |

## O Desafio do Não Supervisionado

> **📌 Nota**
>
> Esta seção corresponde à seção 12.1 de James et al. (2023).

Uma tabela traz, para cada um dos 50 estados americanos, as prisões por três crimes e a parcela da população que vive em áreas urbanas. Outra traz a expressão de 6.830 genes em 64 amostras de células de câncer. Em nenhuma das duas a análise tem uma coluna a prever. O que dá para perguntar a uma tabela assim? E, feita a pergunta, como saber se a resposta está certa?

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("estilo-figuras.mplstyle")

A função `numero_br` escreve os números com a pontuação brasileira:

In [ ]:
numero_br = lambda v, casas=2: (
    f"{v:,.{casas}f}"
    .replace(",", "_")
    .replace(".", ",")
    .replace("_", ".")
    .replace("-", "−")
)

### Uma tabela sem resposta

A tabela `USArrests` é de 1973. Para cada estado, `homicidio`, `agressao` e `estupro` são prisões por esses crimes a cada 100.000 habitantes, e `pop_urbana` é o percentual da população que mora em áreas urbanas. O nome do estado vai para o índice, e cada linha fica com as quatro medidas:

In [ ]:
prisoes = pd.read_csv(
    "dados/USArrests.csv",
    index_col="estado")
n, p = prisoes.shape
print("estados:", n)
print("colunas:", p)
print("primeira linha:",
    prisoes.index[0])
linha = prisoes.iloc[0]
for coluna, v in linha.items():
    print(coluna, numero_br(v, 1))

> **🔧 Função**
>
> - **`pd.read_csv(caminho, index_col="estado")`** — lê o arquivo e usa a coluna `estado` como índice: os nomes dos estados passam a identificar as linhas e deixam de ser uma coluna de dados.
> - **`df.shape`** — a tupla `(linhas, colunas)` da tabela.
> - **`df.index`** — os rótulos das linhas; `df.index[0]` é o rótulo da primeira.
> - **`df.iloc[0]`** — a linha na posição 0, a primeira, como uma série com um valor por coluna.
> - **`serie.items()`** — percorre os pares (rótulo, valor) da série.

São 50 estados e 4 colunas. A primeira linha é a do Alabama, com 13,2 prisões por homicídio, 236 por agressão e 58% da população em áreas urbanas. As colunas não estão na mesma escala. A média, o desvio padrão e a variância de cada uma:

In [ ]:
resumo = pd.DataFrame({
    "média": prisoes.mean(),
    "desvio padrão": prisoes.std(),
    "variância": prisoes.var(),
})
variancias = resumo["variância"]
print("maior variância:",
    variancias.idxmax())
print("menor variância:",
    variancias.idxmin())
tabela = resumo.style
tabela = tabela.set_uuid("resumo")
tabela.format(decimal=",",
    thousands=".", precision=2)

> **🔧 Função**
>
> - **`pd.DataFrame({nome: serie, ...})`** — monta uma tabela a partir de um dicionário: cada chave vira o nome de uma coluna, e cada série, os valores dela, alinhados pelo rótulo das linhas.
> - **`df.mean()`**, **`df.std()`** e **`df.var()`** — a média, o desvio padrão e a variância de cada coluna. A variância do `pandas` divide a soma dos quadrados dos desvios por $n - 1$, e não por $n$; o desvio padrão é a raiz quadrada dela.
> - **`serie.idxmax()`** e **`serie.idxmin()`** — o rótulo do maior e do menor valor da série.
> - **`df.style.format(...)`** — mostra a tabela com vírgula decimal, ponto de milhar e duas casas, sem mudar os números guardados.
> - **`set_uuid(nome)`** — fixa o identificador da tabela no HTML.

As três colunas de crime estão na mesma unidade, e mesmo assim a variância de `agressao`, 6.945,17, é a maior das quatro, e a de `homicidio`, 18,97, a menor: a primeira é cerca de 366 vezes a segunda. A média de `agressao`, 170,76, é quase 22 vezes a de `homicidio`, 7,79, e o desvio padrão de `agressao`, 83,34, é cerca de 19 vezes o de `homicidio`, 4,36; como a variância é o quadrado do desvio padrão, a razão entre as variâncias é exatamente o quadrado da razão entre os desvios. Já `pop_urbana` está em outra unidade, o percentual, e a variância muda quando a unidade muda; a seção 15.3 mostra o que essas diferenças de escala fazem com o resumo da tabela.

Nenhuma dessas colunas é uma resposta. No aprendizado supervisionado, cada observação traz os preditores e uma resposta $y_i$, e a previsão se confere contra ela, como na seção 7.5. Aqui não há $y_i$: não se quer prever as prisões por estupro a partir das outras três colunas, nem a parcela urbana a partir das prisões por crime. As quatro colunas são tratadas do mesmo jeito.

> **🔷 Conceito**
>
> No **aprendizado não supervisionado**, há $n$ observações e, para cada uma, $p$ variáveis $X_1, X_2, \dots, X_p$, sem resposta $Y$. No `USArrests`, $n = 50$ e $p = 4$. Os objetivos mais comuns são dois, e são os deste capítulo:
>
> - **visualizar e resumir** os dados com poucas variáveis novas, que guardem o máximo possível da variação das originais: as **componentes principais**, das seções 15.2 e 15.3;
> - **encontrar subgrupos** de observações parecidas entre si: o **agrupamento** (*clustering*), das seções 15.4 a 15.6.

No `USArrests`, o segundo objetivo vira uma pergunta concreta: os 50 estados se separam em poucos grupos, cada um com estados de perfil parecido nas quatro colunas? Ninguém anotou de antemão a que grupo cada estado pertence. Primeiro, porém, vale olhar a tabela.

Com $p$ variáveis, uma primeira forma de olhar é desenhar um gráfico de dispersão para cada par de colunas. São $p(p-1)/2$ pares. Antes de calcular: um estado com mais prisões por homicídio tende a ser mais urbano? E a ter mais prisões por agressão? A correlação $r$ de cada par, em ordem decrescente:

In [ ]:
from itertools import (
    combinations)

correlacoes = prisoes.corr()
colunas = prisoes.columns
duplas = combinations(colunas, 2)
pares = pd.Series({
    (a, b): correlacoes.loc[a, b]
    for a, b in duplas
})
pares = pares.sort_values(
    ascending=False)
print("pares:", len(pares))
for dupla, r in pares.items():
    a, b = dupla
    print(f"{a} × {b}:",
        numero_br(r))

> **🔧 Função**
>
> - **`df.corr()`** — a tabela das correlações entre as **colunas**, duas a duas; na linha `a` e na coluna `b` fica a correlação entre `a` e `b`.
> - **`df.columns`** — os nomes das colunas da tabela.
> - **`combinations(colunas, 2)`**, do módulo `itertools` — cada par de itens distintos, sem repetir o par na ordem inversa.
> - **`df.loc[a, b]`** — o valor na linha de rótulo `a` e na coluna de rótulo `b`.
> - **`pd.Series({rotulo: valor, ...})`** — uma série a partir de um dicionário. Aqui cada rótulo é o par de colunas, e por isso `pares.items()` entrega o par, em `dupla`, e a correlação, em `r`; `a, b = dupla` separa os dois nomes do par.
> - **`serie.sort_values(...)`** — com `ascending=False`, a série em ordem decrescente de valor.

Os seis gráficos, na ordem da lista, com o Alabama em destaque:

In [ ]:
# Figura: Os 50 estados do `USArrests` em cada par das quatro colunas, um ponto por estado, com a correlação no título. `homicidio`, `agressao` e `estupro` estão em prisões por 100.000 habitantes, e `pop_urbana` em percentual. O ponto laranja, maior, é o Alabama. Os painéis seguem a ordem decrescente da correlação, da esquerda para a direita e de cima para baixo.
fig, eixos = plt.subplots(
    3, 2, figsize=(3.8, 5.4))
for ax, dupla in zip(
        eixos.flat, pares.index):
    a, b = dupla
    r = pares[dupla]
    x, y = prisoes[a], prisoes[b]
    ax.scatter(x, y, s=8,
        color="C0", alpha=0.8,
        linewidths=0)
    ax.scatter(x["Alabama"],
        y["Alabama"], s=28,
        color="C1", zorder=3)
    ax.set_xlabel(a)
    ax.set_ylabel(b)
    titulo = "r = " + numero_br(r)
    ax.set_title(titulo)
    ax.locator_params(nbins=3)
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> - **`plt.subplots(3, 2, figsize=(3.8, 5.4))`** — uma figura com uma grade de 3 linhas e 2 colunas de gráficos, com 3,8 × 5,4 polegadas; **`eixos.flat`** percorre os seis gráficos da grade em ordem de leitura, e `pares.index`, os seis pares na ordem da lista; `pares[dupla]` é a correlação do par.
> - **`ax.scatter(x, y, s, color, alpha)`** — um ponto por par $(x, y)$; `s` é o tamanho do ponto, `alpha`, a opacidade, e `linewidths=0` tira o contorno do ponto. O segundo `scatter` desenha só o Alabama, e `zorder=3` o põe por cima dos outros.
> - **`ax.set_xlabel(texto)`**, **`ax.set_ylabel(texto)`** e **`ax.set_title(texto)`** — o rótulo do eixo horizontal, o do vertical e o título do gráfico.
> - **`ax.locator_params(nbins=3)`** — deixa poucas marcas em cada eixo, para os números não se encostarem num gráfico pequeno; **`ax.set_xlim(left=0)`** e **`ax.set_ylim(bottom=0)`** começam os dois eixos no zero, já que nenhuma das quatro medidas é negativa.
> - **`plt.tight_layout()`** — ajusta o espaço entre os gráficos para os rótulos não se sobreporem; **`plt.show()`** mostra a figura.

Mais urbano, não: `homicidio` e `pop_urbana` têm correlação de 0,07, a menor dos seis pares, e o último painel é uma nuvem quase sem inclinação. Mais prisões por agressão, sim: no primeiro painel, `homicidio` e `agressao`, com 0,80, sobem juntos. As três colunas de crime têm, entre si, as três maiores correlações da lista.

Seis gráficos ainda se leem, mas cada um mostra só duas das quatro colunas, e nenhum usa as quatro medidas de cada estado ao mesmo tempo: o Alabama vira seis pontos laranja, um em cada painel. Dá para reunir os 50 estados num gráfico só, que guarde o máximo possível da variação das quatro colunas? É essa a pergunta das componentes principais.

### Como conferir sem resposta

Dentro de cada célula, cada gene está mais ou menos ativo, e a **expressão** do gene mede essa atividade. A tabela `NCI60` traz a expressão de 6.830 genes, obtida por microarranjo, um chip que mede milhares de genes de uma vez, em 64 amostras de células de câncer. Cada amostra vem de uma linhagem, uma cultura de células mantida em laboratório, e a coluna `tipo` diz, para quase todas, o tipo de câncer dela:

In [ ]:
arquivo = "dados/NCI60.csv.gz"
nci = pd.read_csv(arquivo)
tipo = nci["tipo"]
genes = nci.drop(columns="tipo")
n, p = genes.shape
print("amostras:", n)
print("genes:", numero_br(p, 0))
pares_nci = p * (p - 1) // 2
print("pares de genes:",
    numero_br(pares_nci, 0))
contagens = tipo.value_counts()
print(contagens.to_string())

> **🔧 Função**
>
> - **`pd.read_csv(arquivo)`** — com um arquivo terminado em `.gz`, descomprime antes de ler, pela extensão.
> - **`df.drop(columns="tipo")`** — a tabela sem a coluna `tipo`; a original não muda.
> - **`serie.value_counts()`** — quantas vezes cada valor aparece, do mais frequente ao menos frequente.
> - **`serie.to_string()`** — a série como texto, uma linha por rótulo.

Os rótulos terminados em `-repro` são medições repetidas de duas linhagens, a K562, de leucemia, e a MCF7, de mama; `desconhecido` é uma amostra de origem não identificada. Com $p = 6{.}830$ genes, os gráficos de dispersão, um por par de genes, seriam 23.321.035, demais para olhar um a um.

As 64 amostras se separam em grupos de perfil de expressão parecido? É a pergunta do agrupamento, e ela se faz só com os genes: os métodos das próximas seções recebem a tabela `genes`, e a coluna `tipo` fica de fora. Ela serve para conferir depois o que eles acharem: se um agrupamento juntar as 6 amostras de leucemia num grupo, isso se vê comparando os grupos com `tipo`. Se os grupos não coincidirem com os tipos, o método errou?

Não necessariamente. O tipo de câncer é uma classificação pelo tecido de origem, feita fora da tabela; duas linhagens de tecidos diferentes podem ter perfis de expressão parecidos, e linhagens do mesmo tecido podem diferir. A comparação com `tipo` mostra se os grupos têm relação com uma informação externa, não se estão certos. Essa é a dificuldade do aprendizado não supervisionado. Num problema supervisionado, a previsão se confere contra a resposta em observações que não entraram no ajuste, com um conjunto de teste ou com a validação cruzada do capítulo 10. Sem $Y$, não existe um erro de teste a medir, nem um critério aceito por todos para dizer que um resumo ou um agrupamento está certo. Lado a lado:

| Aspecto | Supervisionado | Não supervisionado |
|---|---|---|
| Dados | preditores e $y_i$ | só as $p$ variáveis |
| Avalia | erro de teste | sem erro de teste |
| Uso | prever `tipo` com `genes` | agrupar por `genes` |

> **🔷 Conceito**
>
> Sem resposta, a análise não supervisionada costuma ser **exploratória**: um passo para conhecer os dados e levantar hipóteses, com escolhas que dependem de quem analisa.

Há, ainda assim, formas parciais de conferir, como ver se os grupos resistem a perder parte dos dados, e a seção 15.6 mostra algumas delas.

## Componentes Principais

> **📌 Nota**
>
> Esta seção corresponde às seções 12.2.1 e 12.2.2 de James et al. (2023).

Cada estado do `USArrests` é um ponto em quatro dimensões: prisões por homicídio, por agressão e por estupro a cada 100.000 habitantes, e o percentual da população em áreas urbanas. Uma página tem duas. Escolher um par de colunas descarta as outras duas por inteiro; um resumo melhor combinaria as quatro em dois números por estado, escolhidos para perder o mínimo da variação. Que combinações são essas, e quanto elas perdem?

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("estilo-figuras.mplstyle")

A função `numero_br` escreve os números com a pontuação brasileira:

In [ ]:
numero_br = lambda v, casas=2: (
    f"{v:,.{casas}f}"
    .replace(",", "_")
    .replace(".", ",")
    .replace("_", ".")
    .replace("-", "−")
)

### A direção de maior variância

Com duas colunas, dá para ver a ideia inteira num plano. As duas aqui são `homicidio` e `agressao`, padronizadas: cada coluna menos a média dela, dividida pelo desvio padrão. As duas ficam com média 0 e variância 1, e nenhuma pesa mais só por estar em números maiores.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import (
    StandardScaler)

prisoes = pd.read_csv(
    "dados/USArrests.csv",
    index_col="estado")
colunas = ["homicidio", "agressao"]
padronizador = StandardScaler()
duas = padronizador.fit_transform(
    prisoes[colunas])
r = np.corrcoef(duas.T)[0, 1]
print("forma:", duas.shape)
medias = duas.mean(axis=0)
variancias = duas.var(axis=0)
print("médias 0:",
    bool(np.allclose(medias, 0)))
print("variâncias 1:",
    bool(np.allclose(variancias, 1)))
print("correlação:", numero_br(r, 3))

> **🔧 Função**
>
> - **`StandardScaler()`** — o padronizador: subtrai de cada coluna a média dela e divide pelo desvio padrão. O desvio padrão dele divide por $n$, e não por $n - 1$ como o `df.std()` do `pandas`; por isso a variância das colunas padronizadas sai 1, a menos do arredondamento.
> - **`padronizador.fit_transform(df)`** — calcula a média e o desvio padrão de cada coluna de `df` e já devolve as colunas padronizadas, como um array do `numpy`.
> - **`arr.mean(axis=0)`** e **`arr.var(axis=0)`** — a média e a variância de cada coluna do array; a variância do `numpy` divide por $n$.
> - **`np.allclose(a, b)`** — `True` quando todos os valores de `a` estão perto dos de `b` (ou do número `b`), com uma tolerância pequena para o arredondamento das contas; `bool(...)` o imprime como `True` ou `False`.
> - **`np.corrcoef(arr.T)`** — a tabela das correlações entre as colunas de `arr`; o `.T` transpõe o array, porque o `np.corrcoef` correlaciona linhas. O elemento `[0, 1]` é a correlação entre a primeira coluna e a segunda.

Cada estado é um ponto no plano, e as duas colunas sobem juntas, com correlação de 0,802. Agora tome uma reta qualquer que passe pela origem, isto é, pela média dos 50 estados. Cada ponto tem um pé sobre ela: o ponto da reta mais perto dele, onde cai o segmento perpendicular. A posição do pé ao longo da reta é a **projeção** do ponto sobre ela, um número só por estado, que resume as duas colunas numa. Algumas retas espalham bem esses 50 números, outras os amontoam. Qual reta os espalha mais? E essa reta fica perto ou longe dos pontos?

Uma reta pela origem fica definida pelo ângulo $\theta$ que faz com o eixo horizontal, ou pelo vetor $v = (\cos\theta, \operatorname{sen}\theta)$, que aponta ao longo dela. Esse vetor tem comprimento 1, porque $\cos^2\theta + \operatorname{sen}^2\theta = 1$, e com um vetor de comprimento 1 a projeção é o produto escalar da seção 4.1: o produto escalar de um ponto com $v$ mede quanto o ponto avança na direção de $v$. O código gira a reta de grau em grau, de 0 a 179, e em cada ângulo mede duas coisas: a variância das posições ao longo da reta e a média dos quadrados das distâncias dos pontos até ela.

In [ ]:
graus = np.arange(180)
variancia = []
distancia = []
for grau in graus:
    rad = np.radians(grau)
    v = np.array([np.cos(rad),
        np.sin(rad)])
    posicao = duas @ v
    pe = np.outer(posicao, v)
    sobra = duas - pe
    d2 = (sobra**2).sum(axis=1)
    variancia.append(posicao.var())
    distancia.append(d2.mean())
variancia = np.array(variancia)
distancia = np.array(distancia)
for grau in [0, 45, 90, 135]:
    print(f"{grau:>3}°:",
        numero_br(variancia[grau], 3),
        "·",
        numero_br(distancia[grau], 3))
print("maior variância em:",
    graus[variancia.argmax()], "graus")
print("menor distância em:",
    graus[distancia.argmin()], "graus")
soma = variancia + distancia
print("soma sempre 2:",
    bool(np.allclose(soma, 2)))

> **🔧 Função**
>
> - **`np.arange(180)`** — os inteiros de 0 a 179.
> - **`np.radians(grau)`**, **`np.cos`** e **`np.sin`** — o ângulo convertido de graus para radianos, e o cosseno e o seno dele.
> - **`np.array(lista)`** — um array do `numpy` com os valores da lista: aqui, o vetor `v`, de duas coordenadas, e depois as listas `variancia` e `distancia`, para que se possa fazer conta com elas.
> - **`duas @ v`** — o produto escalar de cada linha de `duas` com `v`: a projeção de cada ponto, a posição do pé dele ao longo da reta. **`np.outer(posicao, v)`** — cada posição vezes o vetor `v`: as coordenadas dos pés no plano, uma linha por estado.
> - **`(sobra**2).sum(axis=1)`** — o quadrado da distância de cada ponto até o seu pé, somando as duas coordenadas de cada linha.
> - **`arr.argmax()`** e **`arr.argmin()`** — a posição do maior e do menor valor do array.

Em cada linha, o primeiro número é a variância das posições e o segundo, a distância média ao quadrado. A reta de 45 graus, a diagonal, espalha mais os pontos, com variância 1,802, e é também a que passa mais perto deles, com distância média ao quadrado de 0,198. Na de 135 graus, os dois números trocam de lugar: variância 0,198 e distância 1,802.

In [ ]:
# Figura: Em cima, os 50 estados em `homicidio` e `agressao`, as duas padronizadas, com os dois eixos na mesma escala: à esquerda, a reta de 45 graus, a de maior variância, e à direita, tracejada, a de 135 graus. Cada segmento cinza liga um estado ao pé dele sobre a reta. O ponto verde é o Alabama, com o triângulo formado pela origem, pelo pé e pelo ponto. Embaixo, para cada reta pela origem, do ângulo 0 ao 179, a variância das posições ao longo dela, em laranja, e a distância média ao quadrado dos pontos até ela, em cinza, com marcas nos ângulos 0, 45, 90 e 135.
import matplotlib.pyplot as plt

virgula = lambda v, _: (
    f"{v:g}".replace(".", ",")
    .replace("-", "−"))
fig, eixos = plt.subplot_mosaic(
    [["45", "135"], ["giro", "giro"]],
    figsize=(3.8, 4.6),
    height_ratios=[1.1, 1])
alabama = duas[0]
for grau in [45, 135]:
    ax = eixos[str(grau)]
    rad = np.radians(grau)
    v = np.array([np.cos(rad),
        np.sin(rad)])
    pe = np.outer(duas @ v, v)
    for ponto, alvo in zip(duas, pe):
        ax.plot(
            [ponto[0], alvo[0]],
            [ponto[1], alvo[1]],
            color="0.45", linewidth=0.6)
    tipo = "-" if grau == 45 else "--"
    ax.axline((0, 0), v, color="C1",
        linestyle=tipo)
    ax.scatter(duas[:, 0], duas[:, 1],
        s=8, color="C0", zorder=3,
        linewidths=0)
    pe_a = (alabama @ v) * v
    canto = [(0, 0), pe_a, alabama,
        (0, 0)]
    ax.plot(*np.array(canto).T,
        color="C2", linewidth=1.5,
        zorder=4)
    ax.scatter(*alabama, s=30,
        color="C2", zorder=5)
    ax.set_xlim(-2.6, 2.6)
    ax.set_ylim(-2.6, 2.6)
    ax.set_xticks([-2, 0, 2])
    ax.set_yticks([-2, 0, 2])
    for eixo in [ax.xaxis, ax.yaxis]:
        eixo.set_major_formatter(
            virgula)
    ax.set_aspect("equal")
    ax.set_title(f"reta de {grau}°")
    ax.set_xlabel("homicidio")
eixos["45"].set_ylabel("agressao")

ax_giro = eixos["giro"]
ax_giro.plot(graus, variancia,
    color="C1", linewidth=2)
ax_giro.plot(graus, distancia,
    color="0.45", linewidth=2)
marcas = [0, 45, 90, 135]
ax_giro.plot(marcas,
    variancia[marcas], "o",
    color="C1")
ax_giro.plot(marcas,
    distancia[marcas], "o",
    color="0.45")
ax_giro.text(45, 1.95, "variância",
    ha="center", color="C1")
ax_giro.text(135, 1.95, "distância²",
    ha="center", color="0.45")
ax_giro.set_ylim(0, 2.3)
ax_giro.set_xlim(0, 180)
ax_giro.set_xticks([0, 45, 90, 135, 180])
ax_giro.set_yticks([0, 0.5, 1, 1.5, 2])
ax_giro.yaxis.set_major_formatter(
    virgula)
ax_giro.set_xlabel("ângulo da reta (graus)")
ax_giro.set_ylabel("valor (padronizado²)")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> - **`plt.subplot_mosaic(desenho, figsize, height_ratios)`** — uma figura com os gráficos dispostos como no `desenho`, uma lista de linhas com um nome por posição; um nome repetido ocupa as duas posições, como o `"giro"`, que atravessa a largura. `height_ratios` dá as alturas relativas das linhas. Devolve um dicionário: `eixos["45"]` é o gráfico de nome `"45"`.
> - **`ax.axline(ponto, direcao, linestyle)`** — a reta que passa por `ponto` na `direcao` dada, de ponta a ponta do gráfico; `linestyle="--"` a desenha tracejada.
> - **`ax.plot(xs, ys)`** — uma linha pelos pontos `(xs[k], ys[k])`; com dois pontos, um segmento. `ax.plot(*np.array(canto).T)` passa as coordenadas horizontais e as verticais dos três cantos do triângulo, com a origem repetida no fim para fechar o contorno, e com `"o"` a função desenha só os pontos, sem a linha.
> - **`ax.set_aspect("equal")`** — a mesma escala nos dois eixos, para que a perpendicular pareça perpendicular.
> - **`ax.text(x, y, texto)`** — escreve `texto` na posição `(x, y)` dos eixos; `ha="center"` o centra nessa posição. **`ax.set_xticks(lista)`** e **`ax.set_yticks(lista)`** — as marcas dos eixos.
> - **`ax.xaxis.set_major_formatter(funcao)`** — escreve cada marca do eixo com a função dada, que recebe o valor e a posição; a `virgula` troca o ponto decimal pela vírgula.

Pelo teorema de Pitágoras, em cada estado o quadrado da distância até a origem é a posição do pé ao quadrado mais o quadrado da distância até a reta: é o triângulo verde do Alabama, em que a hipotenusa não muda com a reta e os dois catetos, sim. Na média dos 50 estados, o quadrado da distância até a origem vale 1 mais 1, a soma das variâncias das duas colunas, qualquer que seja a reta. Como as duas colunas têm média zero, as posições também têm, e a média das posições ao quadrado é a variância delas; por isso, em todo ângulo, a variância e a distância média ao quadrado somam 2, e **a direção de maior variância é também a reta mais próxima dos pontos**.

A `PCA`, do `scikit-learn`, acha essa direção sem girar reta nenhuma. Ela devolve o vetor de comprimento 1 ao longo da reta, e as coordenadas desse vetor são as **cargas** da componente; a posição de cada estado ao longo da reta, a projeção, é o **escore** do estado na componente:

In [ ]:
from sklearn.decomposition import PCA

pca2 = PCA().fit(duas)
cargas2 = pca2.components_
c1, c2 = cargas2
rad = np.arctan2(
    cargas2[:, 1], cargas2[:, 0])
angulos = np.degrees(rad) % 180
print("retas da 1.ª e da 2.ª:")
print(" ", numero_br(angulos[0], 1),
    "e", numero_br(angulos[1], 1),
    "graus")
print("|cargas| da 1.ª:",
    *[numero_br(v, 3)
        for v in np.abs(c1)])
print("|cargas| da 2.ª:",
    *[numero_br(v, 3)
        for v in np.abs(c2)])
sinal = np.sign(cargas2)
print("mesmo sinal na 1.ª:",
    bool(sinal[0, 0] == sinal[0, 1]))
print("sinais opostos na 2.ª:",
    bool(sinal[1, 0] != sinal[1, 1]))
prop = pca2.explained_variance_ratio_
print("proporção da variância:",
    *[numero_br(v, 3) for v in prop])

> **🔧 Função**
>
> - **`PCA()`**, do módulo `sklearn.decomposition` — a análise de componentes principais; sem argumento, calcula todas as componentes, tantas quanto as colunas. A `PCA` centra cada coluna na média, mas **não** divide pelo desvio padrão: quem padroniza é o `StandardScaler`.
> - **`PCA().fit(duas)`** — calcula as componentes de `duas`. **`pca.components_`** — as cargas, uma **linha** por componente e uma coluna por variável; `components_[0]` é a direção da primeira componente.
> - **`np.arctan2(y, x)`** — o ângulo, em radianos, do vetor `(x, y)`, um para cada linha de `cargas2`; **`np.degrees`** o converte em graus, e `% 180` fica com o resto da divisão por 180, que é o mesmo para a direção e para a oposta.
> - **`np.abs(arr)`** — o valor absoluto de cada elemento. **`np.sign(arr)`** — 1, −1 ou 0, conforme o sinal de cada elemento.
> - **`pca.explained_variance_ratio_`** — a proporção da variância total que cada componente carrega; a seção 15.3 trata dela.

A reta da `PCA` é a de 45 graus, a mesma que o giro de grau em grau achou. As cargas da primeira componente têm o mesmo tamanho, 0,707, e o mesmo sinal: o escore de cada estado é, a menos da escala e do sinal (que é arbitrário, como se vê adiante), a média de `homicidio` e `agressao` padronizadas. Um estado com as duas medidas acima da média fica de um lado da origem, tanto mais longe quanto mais acima; um com as duas abaixo, do outro.

A primeira componente carrega uma proporção de 0,901 da variância: é a variância ao longo da reta, 1,802, dividida pela variância total, 2.

A **segunda componente** é a direção de maior variância entre as que não têm correlação com a primeira. Com duas colunas, isso a deixa perpendicular à primeira, na reta de 135 graus, com cargas de mesmo tamanho e sinais opostos. O escore dela é, a menos da escala, a diferença entre `homicidio` e `agressao` padronizadas, e ela carrega o resto da variância, 0,099.

### As quatro colunas

Com as quatro colunas padronizadas, `pop_urbana`, que está em percentual, passa a ter a mesma escala das três de crime, e a `PCA` acha quatro componentes. As duas primeiras se chamam `PC1` e `PC2`, do inglês *principal component*. As cargas delas, com cada valor passado pela `numero_br`:

In [ ]:
from sklearn.pipeline import (
    make_pipeline)

pca = make_pipeline(
    StandardScaler(), PCA())
escores = pca.fit_transform(prisoes)
nomes = ["PC1", "PC2", "PC3", "PC4"]
cargas = pd.DataFrame(
    pca[-1].components_.T,
    index=prisoes.columns,
    columns=nomes)
tabela = cargas[["PC1", "PC2"]].style
tabela = tabela.set_uuid("cargas")
tabela.format(
    lambda v: numero_br(v, 3))

> **🔧 Função**
>
> - **`make_pipeline(StandardScaler(), PCA())`** — encadeia os dois passos: padroniza as colunas e calcula as componentes das colunas padronizadas.
> - **`pca.fit_transform(prisoes)`** — ajusta os dois passos e devolve os escores, um array com uma linha por estado e uma coluna por componente.
> - **`pca[-1]`** — o último passo do *pipeline*, a `PCA`; `pca[-1].components_.T` tem uma linha por variável e uma coluna por componente, a forma da tabela.
> - **`pd.DataFrame(arr, index=..., columns=...)`** — uma tabela a partir de um array, com os rótulos das linhas em `index` e os nomes das colunas em `columns`.
> - **`tabela.format(funcao)`** — escreve cada valor da tabela como o texto que `funcao` devolve para ele; aqui, a `numero_br` com três casas.

#### Os pesos de uma componente

O escore é uma soma das colunas com pesos, e os pesos são as cargas.

> **🟩 Exemplo**
>
> Com as duas colunas, as cargas da primeira componente tinham o mesmo tamanho, 0,707, e o mesmo sinal. A menos do sinal, o escore do estado $i$ é
>
> $$
> z_{i1} = 0{,}707\,x_{i1} + 0{,}707\,x_{i2},
> $$
>
> em que $x_{i1}$ é o `homicidio` do estado e $x_{i2}$ é a `agressao` dele, as duas padronizadas.

Os índices dizem de quem é cada número: $i$ é a observação (aqui, o estado), $j$ é a coluna e $m$ é a componente. Assim, $x_{ij}$ é o valor da coluna $j$ na observação $i$, já centrado; no exemplo, $x_{i1}$ é o `homicidio` e $x_{i2}$, a `agressao`.

A carga da coluna $j$ na componente $m$ é $\phi_{jm}$, e o escore da observação $i$ na componente $m$ é $z_{im}$; no exemplo, $\phi_{11} = \phi_{21} = 0{,}707$. Com $p$ colunas, a soma da primeira componente, $m = 1$, tem $p$ termos:

$$
z_{i1} = \phi_{11}x_{i1} + \phi_{21}x_{i2} + \dots + \phi_{p1}x_{ip}.
$$

No `USArrests`, $p = 4$, e o escore do Alabama, a primeira linha, é a linha padronizada dele multiplicada pelas cargas da `PC1`, na conta do produto escalar:

In [ ]:
padr = StandardScaler().fit_transform(
    prisoes)
a_mao = padr[0] @ cargas["PC1"]
# + 0.0 troca −0,0 por 0,0
dois = lambda v: numero_br(
    round(v, 2) + 0.0)
print("Alabama (hom·agr·urb·est):")
print(" ", *[dois(v)
    for v in padr[0]])
print("|escore| à mão:",
    numero_br(abs(a_mao), 3))
print("igual ao da PCA:",
    bool(np.isclose(a_mao,
        escores[0, 0])))

> **🔧 Função**
>
> - **`padr[0]`** — a primeira linha do array, a do Alabama, com as quatro colunas na ordem do arquivo; `padr[0] @ cargas["PC1"]` soma os quatro produtos valor vezes carga.
> - **`np.isclose(a, b)`** — `True` quando os dois números estão perto, com a mesma tolerância do `np.allclose`.

O produto dá 0,986 em valor absoluto, o mesmo escore que a `PCA` devolve para o Alabama: o escore é essa soma de valores com pesos. O número sai em valor absoluto porque o sinal da componente é arbitrário; o booleano compara os dois com sinal.

As cargas são as que dão aos $z_{i1}$ a maior variância, com a soma dos quadrados das cargas igual a 1:

$$
\max_{\phi_{11}, \dots, \phi_{p1}}
\frac{1}{n}\sum_{i=1}^{n} z_{i1}^2
\quad\text{com}\quad
\sum_{j=1}^{p}\phi_{j1}^2 = 1.
$$

É a condição de comprimento 1 do vetor $v$ do giro, agora com $p$ coordenadas. Como as colunas estão centradas, os $z_{i1}$ têm média zero, e a média dos quadrados é a variância deles. Sem a condição sobre as cargas, bastaria multiplicá-las por 10 para multiplicar a variância por 100, e o problema não teria solução.

> **🔷 Conceito**
>
> A **primeira componente principal** de $X_1, \dots, X_p$, centradas, é a combinação $Z_1 = \phi_{11}X_1 + \dots + \phi_{p1}X_p$, com $\sum_j\phi_{j1}^2 = 1$, de maior variância. As **cargas** $\phi_{j1}$ são os pesos da combinação, e o valor $z_{i1}$ na observação $i$ é o **escore** da observação: a posição do ponto $i$ ao longo da direção $(\phi_{11}, \dots, \phi_{p1})$.
>
> A segunda componente é a de maior variância entre as combinações sem correlação com a primeira, o que equivale a uma direção perpendicular à primeira; cada componente seguinte faz o mesmo em relação a todas as anteriores.

O sinal de uma componente é arbitrário. O pé de cada ponto sobre a reta é o escore vezes as cargas, o `np.outer(posicao, v)` do giro, e trocar o sinal de todas as cargas troca o sinal de todos os escores sem mudar esse produto: a reta, os pés, as distâncias e a proporção da variância continuam os mesmos. Por isso o que se lê de uma componente é o tamanho das cargas e quais têm o mesmo sinal, nunca se uma carga é positiva ou negativa.

#### A leitura das cargas

Aplicada às cargas das quatro colunas, a regra dá:

In [ ]:
crimes = ["homicidio", "agressao",
    "estupro"]
pc1 = cargas["PC1"].abs()
pc2 = cargas["PC2"].abs()
print("PC1, crimes entre:",
    numero_br(pc1[crimes].min()),
    "e",
    numero_br(pc1[crimes].max()))
print("PC1, |pop_urbana|:",
    numero_br(pc1["pop_urbana"]))
sinais = np.sign(cargas["PC1"])
iguais = sinais == sinais.iloc[0]
print("PC1, as 4 com o mesmo sinal:",
    bool(iguais.all()))
print("PC2, maior |carga|:",
    pc2.idxmax())
print("PC2, |pop_urbana|:",
    numero_br(pc2["pop_urbana"]))
print("PC2, maior |crime|:",
    numero_br(pc2[crimes].max()))
s2 = np.sign(cargas["PC2"])
duas_crime = ["homicidio", "agressao"]
opostos = (s2[duas_crime]
    != s2["pop_urbana"])
print("PC2, hom. e agr. com sinal")
print("  oposto a pop_urbana:",
    bool(opostos.all()))
corr = np.corrcoef(escores.T)
fora = corr[~np.eye(4, dtype=bool)]
print("escores sem correlação:",
    bool(np.all(np.abs(fora) < 1e-10)))

> **🔧 Função**
>
> - **`serie.abs()`** — o valor absoluto de cada elemento da série; **`serie.min()`** e **`serie.max()`** — o menor e o maior valor.
> - **`sinais == sinais.iloc[0]`** — compara cada elemento com o primeiro e devolve uma série de `True` e `False`; **`.all()`** dá `True` quando todos são `True`.
> - **`np.eye(4, dtype=bool)`** — uma matriz 4 × 4 com `True` na diagonal; o `~` a inverte, e `corr[~np.eye(...)]` fica só com as correlações entre componentes diferentes.

Na primeira componente, as três colunas de crime têm cargas entre 0,54 e 0,58, quase o mesmo peso, e `pop_urbana` pesa 0,28, cerca de metade de 0,54. As quatro têm o mesmo sinal. O escore $z_{i1}$ é, então, a menos da escala e do sinal, quase uma média das três colunas de crime padronizadas, com um pouco de urbanização: uma medida do nível geral de prisões do estado.

Na segunda componente, a maior carga em valor absoluto, 0,87, é a de `pop_urbana`, e a maior das três de crime, também em valor absoluto, é 0,42. As cargas de `homicidio` e `agressao` têm sinal oposto ao de `pop_urbana`: o escore $z_{i2}$ mede sobretudo a urbanização, e as prisões por homicídio e por agressão o puxam no sentido contrário. E, como na definição, os escores das quatro componentes não têm correlação entre si.

A `PCA` supõe que a direção em que os dados mais variam é a que mais importa. Um grupo de colunas correlacionadas entre si, mas sem relação com a pergunta que se quer responder, pode dominar a primeira componente, e, sem padronizar, basta uma coluna em números grandes. Cada componente é uma soma das colunas com pesos, uma direção reta: se os pontos se arrumam ao longo de uma curva, a reta e o plano só a aproximam. E nada na escolha das componentes olha para uma resposta: elas resumem só as colunas.

#### Os estados no plano das duas componentes

Com os escores das duas primeiras componentes, cada estado vira um ponto num gráfico só. Antes de ver os nomes: as três colunas de crime têm, na `PC1`, cargas de mesmo sinal e peso parecido. Um estado com muitas prisões pelos três crimes fica perto da origem na horizontal, ou longe dela? Abaixo estão os dois menores e os dois maiores $z_{i1}$, o menor e o maior $z_{i2}$ e o estado mais perto da origem, com as quatro medidas de cada um:

In [ ]:
z = pd.DataFrame(escores[:, :2],
    index=prisoes.index,
    columns=["z1", "z2"])
ordem1 = z["z1"].sort_values()
ordem2 = z["z2"].sort_values()
raio = (z**2).sum(axis=1)
destaques = [
    *ordem1.index[:2],
    *ordem1.index[-2:],
    ordem2.index[0],
    ordem2.index[-1],
    raio.idxmin()]
print("hom. · agr. · est. · urb.")
for estado in destaques:
    # ordem do arquivo: hom, agr, urb, est
    h, a, u, e = prisoes.loc[estado]
    print(f"{estado}:")
    print("  ", numero_br(h, 1),
        "·", numero_br(a, 0),
        "·", numero_br(e, 1),
        "·", numero_br(u, 0))
lado1 = np.sign(z["z1"])
mesmo1 = lado1 == sinais["homicidio"]
lado2 = np.sign(z["z2"])
mesmo2 = lado2 == s2["pop_urbana"]
print("do lado dos crimes:")
print("  Flórida e Nevada:", bool(
    mesmo1[["Flórida", "Nevada"]].all()))
print("do lado de pop_urbana:")
print("  Havaí:", bool(mesmo2["Havaí"]))
lado = np.sign(z.loc["Havaí", "z2"])
topo = (lado * z["z2"]).nlargest(2)
print("lado do Havaí, 2 mais longe:")
for estado, v in topo.items():
    print(" ", estado, numero_br(v))
margem = topo.iloc[0] - topo.iloc[1]
print("  margem:", numero_br(margem))

> **🔧 Função**
>
> - **`.index[:2]`** e **`.index[-2:]`** — os rótulos dos dois primeiros e dos dois últimos da série ordenada.
> - **`(z**2).sum(axis=1)`** — para cada estado, $z_{i1}^2 + z_{i2}^2$, o quadrado da distância até a origem do gráfico.
> - **`serie.nlargest(2)`** — os dois maiores valores da série, do maior para o menor, com os rótulos. Multiplicar `z2` pelo sinal do escore do Havaí faz o lado dele ser o positivo, seja qual for o sinal da componente.

As duas pontas da primeira componente são Dakota do Norte e Vermont, de um lado, e Flórida e Nevada, do outro. A Flórida tem 15,4 prisões por homicídio e 335 por agressão; a Dakota do Norte, 0,8 e 45. Flórida e Nevada ficam do lado para onde apontam as cargas das colunas de crime, e longe da origem: muitas prisões nos três crimes somam três termos grandes do mesmo sinal no escore.

Na segunda componente, o menor e o maior escore são os do Mississípi, com 44% de população urbana, e do Havaí, com 83%, do lado da carga de `pop_urbana`. O Havaí fica nessa ponta por pouco: a Califórnia, logo atrás, está a 0,03 dele. Já o Vermont, com 32%, é menos urbano que o Mississípi e não é ponta. As três linhas, padronizadas:

In [ ]:
tab_padr = pd.DataFrame(padr,
    index=prisoes.index,
    columns=prisoes.columns)
trio = ["Mississípi", "Vermont",
    "Havaí"]
for estado in trio:
    linha = tab_padr.loc[estado]
    h = numero_br(linha["homicidio"])
    a = numero_br(linha["agressao"])
    u = numero_br(linha["pop_urbana"])
    print(f"{estado}:")
    print("  hom.", h, "· agr.", a)
    print("  urb.", u)

O Mississípi junta pouca urbanização, 1,50 desvio padrão abaixo da média, com muitas prisões por homicídio, 16,1, que ficam 1,93 desvio acima dela. Na segunda componente os dois efeitos vão para o mesmo lado, porque `homicidio` e `pop_urbana` têm cargas de sinais opostos. O Vermont é menos urbano, 2,34 desvios abaixo da média, mas também tem poucas prisões por homicídio e por agressão, 1,30 e 1,49 desvio abaixo, e elas o puxam de volta.

E o estado mais perto da origem, a Virgínia? Todos os escores valem zero num estado imaginário com a média de cada coluna. Perto da origem do gráfico quer dizer perto da média nos dois resumos, o nível geral de prisões e a urbanização; se o estado também está perto da média em cada uma das quatro colunas, só a conta diz. Para os dois estados mais perto da origem:

In [ ]:
perto = raio.sort_values().index[:2]
print("mais perto da origem:")
print(" ", *perto)
for estado in perto:
    linha = tab_padr.loc[estado]
    print(f"{estado}, padronizado:")
    for coluna, v in linha.items():
        print(" ", coluna, numero_br(v))
    todas = (linha.abs() < 0.2).all()
    print("  todas a menos de 0,2:",
        bool(todas))
# agressao só com PC1 e PC2:
# escore vezes carga, somado
segundo = perto[1]
z_seg = z.loc[segundo].to_numpy()
phi = cargas.loc["agressao",
    ["PC1", "PC2"]].to_numpy()
agr = tab_padr.loc[segundo,
    "agressao"]
print(f"{segundo}, agressao:")
print("  dado:", numero_br(agr))
print("  só PC1 e PC2:",
    numero_br(z_seg @ phi))

> **🔧 Função**
>
> - **`serie.to_numpy()`** — os valores da série como array do `numpy`, sem os rótulos; o `@` entre duas séries exige os mesmos rótulos nas duas, e `z1`, `z2` não são `PC1`, `PC2`.

As quatro medidas da Virgínia ficam a menos de 0,2 desvio padrão da média de cada coluna. As do Delaware, o segundo estado mais perto da origem, não: a `agressao` dele fica 0,82 desvio acima da média. Só com as duas primeiras componentes, no pé do Delaware sobre o plano do gráfico (escore vezes carga em cada componente, somados, a conta que a última parte desta seção generaliza), ela fica em −0,03: o gráfico, que tem só essas duas, não mostra esse afastamento.

A figura junta os 50 estados e as quatro cargas. Antes de olhar: com os sinais da `PC2`, em que ponta ficaria um estado muito urbano e com poucas prisões?

In [ ]:
# Figura: Os 50 estados do `USArrests` pelos escores das duas primeiras componentes, um ponto azul por estado, com as colunas padronizadas. As setas laranja são as cargas de cada coluna nas duas componentes, multiplicadas por 3 para caber na escala dos escores. Os nomes marcam os dois menores e os dois maiores escores da 1.ª, o menor e o maior da 2.ª e o estado mais perto da origem. Os dois eixos têm a mesma escala; o sinal de cada componente é arbitrário, e trocá-lo espelharia o gráfico sem mudar a leitura.
fig, ax = plt.subplots(
    figsize=(3.8, 4.0))
ax.scatter(z["z1"], z["z2"], s=12,
    color="C0", alpha=0.8,
    linewidths=0)
for estado in destaques:
    x, y = z.loc[estado]
    rotulo = estado.replace(
        " do ", "\ndo ")
    comp = max(np.hypot(x, y), 1e-9)
    passo = (8 * x / comp,
        8 * y / comp)
    ha = "left" if x > 0 else "right"
    va = "bottom" if y > 0 else "top"
    if abs(x) > 2 * abs(y):
        va = "center"
    if abs(y) > 2 * abs(x):
        ha = "center"
    fio = None
    if estado == raio.idxmin():
        passo = (-22, -26)
        fio = dict(arrowstyle="-",
            color="0.45", lw=0.6)
    ax.annotate(rotulo, (x, y),
        xytext=passo,
        textcoords="offset points",
        ha=ha, va=va, fontsize=9,
        arrowprops=fio)
for coluna in cargas.index:
    x, y = 3 * cargas.loc[
        coluna, ["PC1", "PC2"]]
    ax.annotate("", xy=(x, y),
        xytext=(0, 0),
        arrowprops=dict(color="C1",
            arrowstyle="-|>", lw=1.5))
    comp = np.hypot(x, y)
    passo = (8 * x / comp, 8 * y / comp)
    ax.annotate(coluna, (x, y),
        xytext=passo,
        textcoords="offset points",
        ha="left" if x > 0 else "right",
        va="center", fontsize=9,
        color="C1", fontweight="bold")
ax.set_xlim(-4.6, 4.6)
ax.set_ylim(-3.4, 3.4)
ax.set_aspect("equal")
ax.set_xlabel("escore da 1.ª, $z_{i1}$")
ax.set_ylabel("escore da 2.ª, $z_{i2}$")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> - **`ax.annotate(texto, (x, y), xytext, textcoords="offset points")`** — escreve `texto` junto do ponto `(x, y)`, deslocado de `xytext` em pontos tipográficos; `ha` e `va` alinham o texto na horizontal e na vertical. Os nomes, dos estados e das colunas, vão para fora, na direção que liga a origem ao ponto; o `if` centra o nome na vertical quando essa direção é quase horizontal, e na horizontal quando ela é quase vertical; `"\ndo "` quebra os nomes compostos em duas linhas, e `fontweight="bold"` põe os nomes das colunas em negrito. O estado mais perto da origem, no meio de vizinhos, tem o nome mais afastado e ligado ao ponto por um fio cinza: `arrowprops` com `arrowstyle="-"` desenha a linha sem ponta; com `None`, nenhuma linha.
> - **`np.hypot(x, y)`** — o comprimento do vetor `(x, y)`, $\sqrt{x^2 + y^2}$; dividir `x` e `y` por ele dá a direção com comprimento 1.
> - **`ax.annotate("", xy, xytext, arrowprops)`** — sem texto, desenha só uma seta de `xytext` até `xy`; `arrowprops` dá a cor, a ponta (`"-|>"`) e a espessura.

Um gráfico assim, com escores e cargas juntos, se chama *biplot*. As três setas de crime apontam para o mesmo lado do eixo horizontal, e a de `pop_urbana` se afasta delas, sobretudo na vertical: é o retrato das cargas da tabela. A regra de leitura é a da projeção: um estado que fica no sentido de uma seta tende a ter aquela coluna acima da média, e um que fica no sentido oposto, abaixo. O fator 3 muda só o comprimento das setas em relação aos pontos, não a direção delas.

E o estado muito urbano e com poucas prisões ficaria na ponta do Havaí, na vertical. O próprio Havaí é um estado assim: a `pop_urbana` dele fica 1,22 desvio padrão acima da média, e as prisões por homicídio e por agressão, 0,58 e 1,51 desvio abaixo. Na segunda componente, com os sinais opostos, a urbanização e as poucas prisões o empurram para o mesmo lado. Na horizontal, as poucas prisões puxam um estado assim para o lado da Dakota do Norte, e a urbanização, com carga menor, para o outro.

### A melhor aproximação de dimensão $M$

Na reta de 45 graus, a distância de cada estado até o pé dele era o que a reta deixava de fora. Trocar o ponto pelo pé é aproximar duas colunas com um número só, o escore: o pé é o escore vezes as cargas. Com as quatro colunas, o mesmo vale com $M$ componentes: o valor $x_{ij}$, já centrado, se aproxima pela soma das $M$ primeiras componentes, cada escore vezes a carga:

$$
x_{ij} \approx \sum_{m=1}^{M} z_{im}\,\phi_{jm}.
$$

Para o estado $i$ e a coluna $j$, cada termo da soma é o escore do estado na componente $m$ vezes a carga da coluna nessa componente, e a soma percorre as $M$ primeiras componentes. Entre todas as aproximações dessa forma, com quaisquer $n \times M$ números no lugar dos escores e $p \times M$ no lugar das cargas, a de menor soma dos quadrados dos erros,

$$
\sum_{j=1}^{p}\sum_{i=1}^{n}
\Big(x_{ij} - \sum_{m=1}^{M} z_{im}\,\phi_{jm}\Big)^2,
$$

é a das componentes principais. Dentro do parêntese está o erro da aproximação num valor, o da coluna $j$ no estado $i$; o quadrado dele é somado em todos os estados e em todas as colunas. A primeira componente é a reta mais próxima dos pontos, as duas primeiras formam o plano mais próximo, e assim por diante. Quanto a aproximação erra no `USArrests`, com $M$ de 1 a 4?

In [ ]:
erros = []
iguais = []
linhas = {}
for M in range(1, 5):
    pca_m = PCA(n_components=M)
    zs = pca_m.fit_transform(padr)
    aprox = pca_m.inverse_transform(zs)
    produto = zs @ pca_m.components_
    iguais.append(
        np.allclose(aprox, produto))
    erro = ((padr - aprox)**2).mean()
    erros.append(erro)
    linhas[M] = aprox[0]
    print(f"M = {M}:", numero_br(erro, 4))
print("cai a cada M:",
    bool(np.all(np.diff(erros) < 0)))
print("exata com M = 4:",
    bool(np.allclose(aprox, padr)))
print("soma de z·φ, todo M:",
    bool(all(iguais)))
print("Alabama (hom·agr·urb·est):")
print("  dado:",
    *[dois(v) for v in padr[0]])
for M in [1, 2, 4]:
    print(f"  M = {M}:",
        *[dois(v) for v in linhas[M]])

> **🔧 Função**
>
> - **`PCA(n_components=M)`** — calcula só as `M` primeiras componentes; **`fit_transform`** devolve os `M` escores de cada linha.
> - **`pca.inverse_transform(zs)`** — volta dos escores às colunas: soma, em cada linha, os escores vezes as cargas, mais a média de cada coluna, que aqui é zero.
> - **`zs @ pca.components_`** — a soma $\sum_m z_{im}\phi_{jm}$ para todo $i$ e todo $j$ de uma vez.
> - **`np.diff(lista)`** — as diferenças entre valores vizinhos; todas negativas quer dizer que a lista só cai. **`np.all(arr)`** — `True` quando todos os elementos são `True`.

O erro quadrático médio por valor, numa tabela padronizada em que cada coluna tem variância 1, vai de 0,3799 com uma componente a 0,0434 com três, e cai a cada componente. Com $M = 4$ a aproximação é exata: quatro componentes reconstroem as quatro colunas. O booleano da soma confirma, em cada $M$, que a aproximação da `PCA` é a soma dos escores vezes as cargas.

Na linha do Alabama, uma componente só põe o estado acima da média nas quatro colunas, inclusive em `pop_urbana`, em que o dado fica abaixo dela, em −0,53. Com duas componentes, a `pop_urbana` aproximada passa para baixo da média, −0,72, e com as quatro a linha volta a ser a do dado.

As duas combinações que perdem o mínimo são as duas primeiras componentes, e o plano delas é o do *biplot*. O que elas perdem se mede pela aproximação: com $M = 2$, o erro quadrático médio por valor cai para 0,1325, contra 0,3799 com uma componente. Quanto da variação as duas componentes guardam, e quantas componentes guardar, é a pergunta da seção 15.3.

## Proporção da Variância Explicada

> **📌 Nota**
>
> Esta seção corresponde às seções 12.2.3, 12.2.4 e 12.2.5 de James et al. (2023).

Quatro colunas não cabem inteiras num plano. Quanto da variação do `USArrests` fica fora das duas primeiras componentes principais? E o resumo sairia o mesmo se as prisões por agressão fossem contadas a cada 100 habitantes, e não a cada 100.000?

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("estilo-figuras.mplstyle")

A função `numero_br` escreve os números com a pontuação brasileira:

In [ ]:
numero_br = lambda v, casas=2: (
    f"{v:,.{casas}f}"
    .replace(",", "_")
    .replace(".", ",")
    .replace("_", ".")
    .replace("-", "−")
)

### Quanto cada componente carrega

Com as colunas padronizadas, cada uma tem variância 1, e a tabela inteira tem variância total 4, a soma das quatro. Os escores das quatro componentes também têm, cada um, uma variância. Como essas variâncias se comparam com as das colunas?

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition \
    import PCA
from sklearn.pipeline import (
    make_pipeline)
from sklearn.preprocessing \
    import StandardScaler

prisoes = pd.read_csv(
    "dados/USArrests.csv",
    index_col="estado")
n_est = len(prisoes)
padronizador = StandardScaler()
padr = padronizador.fit_transform(
    prisoes)
pca = make_pipeline(
    StandardScaler(), PCA())
escores = pca.fit_transform(
    prisoes)
var_col = padr.var(axis=0)
var_z = escores.var(axis=0)
print("estados:", n_est)
print("variâncias das colunas:")
print(" ", *[numero_br(v)
    for v in var_col])
print("  soma:",
    numero_br(var_col.sum()))
print("variâncias dos escores:")
print(" ", *[numero_br(v)
    for v in var_z])
print("  soma:",
    numero_br(var_z.sum()))
cai = np.all(np.diff(var_z) < 0)
print("  cada uma < a anterior:",
    bool(cai))

As quatro colunas têm variância 1,00 cada; os escores, não: a primeira componente fica com 2,48, e cada uma das seguintes com menos que a anterior. As duas listas somam o mesmo, 4,00. A `PCA` redistribui a variação entre as componentes sem mudar o total, e põe o máximo possível na primeira componente, depois o máximo do que sobra na segunda, e assim por diante.

A primeira componente fica, então, com 2,48 dos 4,00 da variância total, ou 0,62 dela. Essa fração é a **proporção da variância explicada** pela componente, e a `PCA` a devolve pronta, no atributo `explained_variance_ratio_`. A soma das proporções das $M$ primeiras componentes é a **proporção acumulada**:

In [ ]:
passo_pca = pca[-1]
pve = (passo_pca
    .explained_variance_ratio_)
acumulada = np.cumsum(pve)
print("proporção · acumulada")
for m in range(4):
    a = numero_br(pve[m], 4)
    b = numero_br(acumulada[m], 4)
    rotulo = f"PC{m + 1}:"
    print(rotulo, a, "·", b)

> **🔧 Função**
>
> - **`np.cumsum(arr)`** — as somas acumuladas: o elemento $m$ do resultado é a soma dos $m$ primeiros elementos de `arr`.

A primeira componente explica 0,6201 da variância, e as duas primeiras juntas, 0,8675.

> **🔷 Conceito**
>
> Com as colunas centradas, a **variância total** da tabela é a soma das variâncias das $p$ colunas,
>
> $$
> \sum_{j=1}^{p}\frac{1}{n}\sum_{i=1}^{n} x_{ij}^2,
> $$
>
> e a variância da componente $m$ é a dos escores dela,
>
> $$
> \frac{1}{n}\sum_{i=1}^{n} z_{im}^2.
> $$
>
> A **proporção da variância explicada** pela componente $m$, ou PVE (do inglês *proportion of variance explained*), é a razão entre as duas:
>
> $$
> \text{PVE}_m =
> \frac{\sum_{i=1}^{n} z_{im}^2}
> {\sum_{j=1}^{p}\sum_{i=1}^{n} x_{ij}^2}.
> $$
>
> As proporções não são negativas e, somadas todas as componentes, dão 1. A **proporção acumulada** das $M$ primeiras é $\text{PVE}_1 + \dots + \text{PVE}_M$.

O $1/n$ aparece em cima e embaixo e se cancela na razão. A conta, então, sai direto dos escores e da tabela padronizada, sem o divisor:

In [ ]:
num = (escores**2).sum(axis=0)
den = (padr**2).sum()
igual = np.allclose(
    num / den, pve)
print("à mão, igual:",
    bool(igual))

A conta à mão dá as mesmas proporções da `PCA`. Num gráfico das proporções, cada componente vira um ponto:

In [ ]:
# Figura: A proporção da variância explicada por cada uma das quatro componentes do `USArrests`, com as colunas padronizadas (em cima) e a proporção acumulada (embaixo).
virgula = lambda v, _: (
    f"{v:g}".replace(".", ","))
ks = np.arange(1, 5)
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(3.8, 4.4),
    sharex=True)
ax1.plot(ks, pve, "o-",
    color="C0")
ax2.plot(ks, acumulada, "o-",
    color="C0")
ax1.set_ylabel("proporção")
ax2.set_ylabel("acumulada")
ax2.set_xlabel("componente")
ax2.set_xticks(ks)
ax2.set_xlim(0.7, 4.3)
for ax in (ax1, ax2):
    ax.set_ylim(0, 1.05)
    ax.yaxis.set_major_formatter(
        virgula)
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> - **`plt.subplots(2, 1, ..., sharex=True)`** — dois gráficos empilhados, com o mesmo eixo horizontal, um em `ax1` e outro em `ax2`.

Um gráfico da proporção contra o número da componente, como o de cima, se chama *scree plot*, ou gráfico do cotovelo, pela forma que costuma ter: uma queda forte nas primeiras componentes e depois um trecho quase plano.

#### A proporção acumulada e a aproximação

A aproximação de dimensão $M$ troca cada valor $x_{ij}$ pela soma $\sum_{m=1}^{M} z_{im}\phi_{jm}$ das $M$ primeiras componentes. Quanto mais componentes, mais variância a aproximação guarda e menos ela erra. Chame de RSS a soma dos quadrados dos erros da aproximação, em todos os estados e todas as colunas, e de TSS a soma dos quadrados dos valores da tabela, $\sum_j\sum_i x_{ij}^2$. A conta abaixo calcula, para cada $M$, o erro quadrático médio por valor e $1 - \text{RSS}/\text{TSS}$.

O erro da aproximação e a proporção acumulada têm alguma relação?

In [ ]:
cargas = passo_pca.components_
tss = (padr**2).sum()
lista = []
erros = []
cab = "M: erro por valor"
print(cab, "· 1 − RSS/TSS")
for M in range(1, 5):
    zs = escores[:, :M]
    aprox = zs @ cargas[:M]
    resto = padr - aprox
    rss = (resto**2).sum()
    lista.append(1 - rss / tss)
    erros.append(
        (resto**2).mean())
    a = numero_br(erros[-1], 4)
    b = numero_br(lista[-1], 4)
    print(f"M = {M}:", a, "·", b)
igual = np.allclose(lista,
    acumulada)
print("1 − RSS/TSS = acumulada:",
    bool(igual))
media = tss / padr.size
print("média dos x² igual a 1:",
    bool(np.isclose(media, 1)))

> **🔧 Função**
>
> - **`arr.size`** — o número de valores do array, linhas vezes colunas.

A coluna da direita repete a proporção acumulada impressa acima: a proporção acumulada é o complemento do erro relativo da aproximação, $\text{RSS}/\text{TSS}$. A coluna da esquerda é o erro quadrático médio por valor: 0,3799 com uma componente e 0,1325 com duas. Os dois somam 1 em cada linha, e o motivo está no último booleano: numa tabela padronizada, a média dos $x_{ij}^2$ é 1, e o erro por valor, que é RSS dividido pelo número de valores, fica igual a $\text{RSS}/\text{TSS}$. Com três componentes, o erro, 0,0434, é a proporção que só a quarta componente explica, também 0,0434: é o que fica de fora.

Por que a variância total se divide assim? Em cada estado, a origem, o ponto $x_i$ e o pé dele sobre as $M$ primeiras componentes formam um triângulo retângulo, como o triângulo verde do Alabama na primeira figura da seção 15.2. Pelo teorema de Pitágoras, o quadrado da distância do ponto à origem, $\sum_j x_{ij}^2$, é o quadrado da distância do pé à origem mais o quadrado do resto, e o primeiro é $\sum_{m=1}^{M} z_{im}^2$, porque as cargas de cada componente têm comprimento 1 e as direções são perpendiculares entre si. Somado nos $n$ estados e dividido por $n$, isso dá:

$$
\begin{aligned}
\sum_{j=1}^{p}\frac{1}{n}\sum_{i=1}^{n} x_{ij}^2
&= \sum_{m=1}^{M}\frac{1}{n}\sum_{i=1}^{n} z_{im}^2 \\
&\quad + \frac{\text{RSS}}{n}.
\end{aligned}
$$

A variância total, à esquerda, é a variância das $M$ primeiras componentes mais o erro da aproximação. Divididos os três termos pela variância total, $\text{TSS}/n$, o primeiro vira 1, o segundo vira a proporção acumulada e o terceiro, $\text{RSS}/\text{TSS}$.

> **🔷 Conceito**
>
> A proporção acumulada das $M$ primeiras componentes é
>
> $$
> \text{PVE}_1 + \dots + \text{PVE}_M = 1 - \frac{\text{RSS}}{\text{TSS}}
> $$
>
> É o $R^2$ da seção 8.2, com a tabela aproximada no lugar das respostas previstas e a própria tabela no lugar das respostas. Como a variância total não muda, as componentes que guardam a maior variância são as que deixam o menor erro: a direção de maior variância e a reta mais próxima dos pontos coincidem.

### Padronizar ou não

Tudo até aqui usou as colunas padronizadas. A `PCA` sozinha só centra as colunas: cada uma entra com a variância que tem, na unidade em que foi medida. Se uma coluna tem quase toda a variância da tabela, para onde deve apontar a primeira componente?

In [ ]:
variancia = prisoes.var()
total = variancia.sum()
fatia = variancia / total
f_agr = fatia["agressao"]
print("fatia de agressao:",
    numero_br(f_agr, 3))
cru = PCA().fit(prisoes)
abs_cru = np.abs(cru.components_)
nomes = prisoes.columns
c_cru = pd.Series(abs_cru[0],
    index=nomes)
print("maior |carga| na PC1:")
print(" ", c_cru.idxmax(),
    numero_br(c_cru.max(), 3))
pve_cru = (cru
    .explained_variance_ratio_)
print("proporção da PC1:",
    numero_br(pve_cru[0], 3))

> **🔧 Função**
>
> - **`pd.Series(arr, index=nomes)`** — uma `Series` com os valores do array `arr`, cada um rotulado pelo nome correspondente em `nomes`.

`agressao` tem, sozinha, 0,956 da variância total das quatro colunas, e a primeira componente é quase ela: a maior carga, em valor absoluto, é a de `agressao`, 0,995, e a componente explica 0,966 da variância.

A igualdade entre $1 - \text{RSS}/\text{TSS}$ e a proporção acumulada não depende da padronização: o triângulo retângulo vale para qualquer tabela centrada. Na tabela só centrada, com duas componentes:

In [ ]:
cent = prisoes.to_numpy()
cent = cent - cent.mean(axis=0)
z_cru = cru.transform(prisoes)
ap = (z_cru[:, :2]
    @ cru.components_[:2])
rss = ((cent - ap)**2).sum()
tss_cru = (cent**2).sum()
r2 = 1 - rss / tss_cru
acum2 = pve_cru[:2].sum()
print("1 − RSS/TSS:",
    numero_br(r2, 4))
print("acumulada, M = 2:",
    numero_br(acum2, 4))
print("iguais:",
    bool(np.isclose(r2, acum2)))

> **🔧 Função**
>
> - **`cru.transform(prisoes)`** — os escores dos estados nas componentes já calculadas pelo `fit`.

O gráfico dos escores das duas primeiras componentes, com as cargas das colunas em setas, mostra o que a primeira componente virou:

In [ ]:
# Figura: Os 50 estados do `USArrests` pelos escores das duas primeiras componentes, calculados com as colunas só centradas, sem padronizar, um ponto azul por estado. As setas laranja são as cargas de cada coluna nas duas componentes, multiplicadas por 100 para ficarem visíveis na escala dos escores. As de `estupro` e `homicidio` são curtas, junto da origem, e o nome de cada uma se liga à ponta da seta por um fio cinza; a de `homicidio` fica quase toda sob a base da de `agressao`.
fig, ax = plt.subplots(
    figsize=(3.8, 2.8))
ax.scatter(
    z_cru[:, 0], z_cru[:, 1],
    s=12, color="C0", alpha=0.8,
    linewidths=0)
seta = dict(color="C1",
    arrowstyle="-|>", lw=1.5)
fio = dict(arrowstyle="-",
    color="0.45", lw=0.6)
# onde fica o nome de cada seta
lugar = {
    "agressao": (100, -42),
    "pop_urbana": (5, 114),
    "estupro": (-80, 70),
    "homicidio": (-80, -55)}
for j, coluna in enumerate(nomes):
    x, y = 100 * cru.components_[
        :2, j]
    ax.annotate("", xy=(x, y),
        xytext=(0, 0),
        arrowprops=seta)
    curta = np.hypot(x, y) < 50
    ax.annotate(coluna, (x, y),
        xytext=lugar[coluna],
        ha="center", va="center",
        fontsize=9, color="C1",
        fontweight="bold",
        arrowprops=(fio if curta
            else None))
ax.set_xlim(-150, 190)
ax.set_ylim(-75, 135)
ax.set_xticks([-100, 0, 100])
ax.set_aspect("equal")
ax.set_xlabel("escore da 1.ª")
ax.set_ylabel("escore da 2.ª")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> - **`ax.annotate(coluna, (x, y), xytext=lugar[coluna])`** — sem `textcoords`, o `xytext` é uma posição nos eixos do gráfico, e não um deslocamento.

Os estados se espalham quase só na horizontal, ao longo da seta de `agressao`: o gráfico mostra, praticamente, as prisões por agressão de cada estado.

E se `agressao` fosse contada a cada 100 habitantes? É a mesma medida dividida por 1.000, e a variância dela, dividida por 1.000², fica um milhão de vezes menor. A tabela abaixo põe lado a lado a primeira componente em três versões: a da tabela como está (`cru`), a da tabela com `agressao` por 100 habitantes, sem padronizar (`÷ 1.000`), e a das colunas padronizadas (`padr.`):

In [ ]:
por_100 = prisoes.assign(
    agressao=prisoes["agressao"]
    / 1000)
cru100 = PCA().fit(por_100)
c1 = pd.Series(
    np.abs(cru100.components_[0]),
    index=nomes)
print("maior |carga| na PC1,")
print("  ÷ 1.000:", c1.idxmax(),
    numero_br(c1.max(), 3))
j = nomes.get_loc("agressao")
pve100 = (cru100
    .explained_variance_ratio_)
tab = pd.DataFrame({
    "cru": [abs_cru[0, j],
        pve_cru[0]],
    "÷ 1.000": [c1["agressao"],
        pve100[0]],
    "padr.": [abs(cargas[0, j]),
        pve[0]]},
    index=["|carga| agressao",
        "proporção da PC1"])
t = tab.style.set_uuid("escalas")
t.format(
    lambda v: numero_br(v, 3))

> **🔧 Função**
>
> - **`df.assign(agressao=...)`** — uma cópia da tabela com a coluna `agressao` trocada pelos valores dados; a original não muda.
> - **`df.columns.get_loc(nome)`** — a posição da coluna de nome `nome`; aqui, a de `agressao`, para ler a carga dela em `abs_cru` e em `cargas`.

Sem padronizar, a carga de `agressao` na primeira componente cai, em valor absoluto, de 0,995 para 0,002, e quem passa a dominar é `pop_urbana`, com 0,928; a primeira componente explica agora 0,733 da variância, contra 0,966 antes. A mesma tabela, com uma unidade trocada, dá outro resumo. Com as colunas padronizadas, na última coluna, a carga de `agressao` é 0,583 e a proporção, 0,620. A conta abaixo refaz essa versão com `agressao` a cada 100 habitantes: a troca de unidade muda alguma coisa?

In [ ]:
pad100 = make_pipeline(
    StandardScaler(), PCA())
pad100.fit(por_100)
pca100 = pad100[-1]
r100 = (pca100
    .explained_variance_ratio_)
c100 = np.abs(pca100.components_)
iguais_pve = np.allclose(
    r100, pve)
iguais_c = np.allclose(c100,
    np.abs(cargas))
cab = "padronizando, iguais"
print(cab, "a antes:")
print("  proporções:",
    bool(iguais_pve))
print("  |cargas|:",
    bool(iguais_c))

Não muda nada: as proporções e as cargas, em valor absoluto, saem as mesmas de antes, porque padronizar divide cada coluna pelo próprio desvio padrão e apaga a unidade.

Não é desejável que o resumo dependa de uma escolha arbitrária de unidade, e por isso, em geral, as colunas se padronizam antes da `PCA`. No `USArrests` elas estão em unidades diferentes, prisões por 100.000 habitantes e percentual, e padronizar é o caminho. Quando todas as colunas estão na mesma unidade, como é o caso da expressão de milhares de genes medidos do mesmo jeito, pode-se deixá-las sem padronizar, e aí os genes que mais variam pesam mais. A padronização, nesse caso, é uma escolha: ela dá a cada gene o mesmo peso. O `NCI60`, adiante, entra padronizado.

> **📌 Nota**
>
> O sinal de cada componente é arbitrário, como na seção 15.2. A `PCA` desta versão do `scikit-learn` fixa uma convenção, que deixa positiva a maior carga, em valor absoluto, de cada componente, e o booleano confere nas quatro componentes do `USArrests` padronizado:

In [ ]:
abs_c = np.abs(cargas)
maior = abs_c.argmax(axis=1)
topo = cargas[np.arange(4), maior]
print("PCA, maior |carga| > 0:",
    bool((topo > 0).all()))

> **🔧 Função**
>
> - **`abs_c.argmax(axis=1)`** — a posição do maior valor em cada linha; **`cargas[np.arange(4), maior]`** — em cada linha, o elemento da coluna indicada em `maior`.

### Quantas componentes

Quantas componentes guardar? No `USArrests`, as duas primeiras explicam 0,8675 da variância. O gráfico do cotovelo se lê pelas quedas entre componentes vizinhas; o sinal de menos em `-np.diff` faz das diferenças quedas:

In [ ]:
quedas_us = -np.diff(pve)
print("quedas entre vizinhas:")
print(" ", *[numero_br(v, 4)
    for v in quedas_us])
q = quedas_us
print("2.ª queda ÷ 3.ª:",
    numero_br(q[1] / q[2]))

A proporção cai 0,3726 da primeira para a segunda componente, 0,1583 da segunda para a terceira e 0,0458 da terceira para a quarta. Quem põe o cotovelo depois da segunda tem argumento: a primeira queda é grande, e a seguinte, bem menor. Quem o põe depois da terceira também tem: a segunda queda ainda é 3,46 vezes a terceira.

Essa leitura é *ad hoc*: não há um critério objetivo e único para decidir quantas componentes bastam. A resposta depende dos dados e do uso. Na prática, olham-se as primeiras componentes em busca de padrões nos dados, e segue-se olhando enquanto eles aparecerem.

E no `NCI60`, com muito mais genes do que amostras? Antes de ver a saída: a primeira componente vai explicar mais ou menos da variância do que a do `USArrests`, 0,6201? Primeiro, o tamanho da tabela, a escala dos genes e o número de componentes, com os genes padronizados:

In [ ]:
nci = pd.read_csv(
    "dados/NCI60.csv.gz")
genes = nci.drop(columns="tipo")
n, p = genes.shape
print("amostras:", n)
print("genes:", numero_br(p, 0))
v = genes.var()
razao = v.max() / v.min()
print("maior/menor variância:",
    numero_br(razao, 0))
fatia_g = v.max() / v.sum()
print("fatia do mais variável:",
    numero_br(fatia_g, 4))
pca_nci = make_pipeline(
    StandardScaler(), PCA())
pca_nci.fit(genes)
passo_nci = pca_nci[-1]
pve_nci = (passo_nci
    .explained_variance_ratio_)
print("componentes:",
    passo_nci.n_components_)
print("mín(n − 1, p):",
    min(n - 1, p))
print("a 64.ª é quase zero:",
    bool(pve_nci[-1] < 1e-10))
uteis = pve_nci[:63]
soma = uteis.sum()
print("as 63 somam 1:",
    bool(np.isclose(soma, 1)))

> **🔧 Função**
>
> - **`pca_nci[-1].n_components_`** — o número de componentes que a `PCA` calculou.

São 64 amostras e 6.830 genes. Os genes estão todos na mesma unidade, mas a maior variância entre eles é 289 vezes a menor. Nenhum gene sozinho domina a tabela: o mais variável fica com 0,0028 da variância total, longe dos 0,956 de `agressao` no `USArrests`. Ainda assim, sem padronizar, os genes de maior variância pesariam mais nas primeiras componentes. Padronizados, todos entram com o mesmo peso.

A `PCA` devolve 64 componentes, mas a última tem proporção praticamente nula: centradas, as 64 amostras somam zero em cada gene, e uma delas fica determinada pelas outras 63. Com $n$ observações e $p$ colunas, há no máximo $\min(n - 1, p)$ componentes com variância maior que zero, aqui 63, e as proporções delas somam 1. As primeiras proporções, a acumulada e as quedas entre vizinhas:

In [ ]:
cab = "proporção das 5"
print(cab, "primeiras:")
print(" ", *[numero_br(v, 3)
    for v in uteis[:5]])
acum_nci = np.cumsum(uteis)
print("acumulada:")
for M in [7, 20]:
    v = acum_nci[M - 1]
    print(f"  M = {M}:",
        numero_br(v, 3))
quedas = -np.diff(uteis)
k = quedas.argmax() + 1
print(f"maior queda: da {k}.ª",
    f"para a {k + 1}.ª")
resto = quedas[3:].max()
cab = "maior queda da 4.ª"
print(cab, "em diante:",
    numero_br(resto, 4))
k4 = quedas[3:].argmax() + 4
print(f"  é da {k4}.ª para a",
    f"{k4 + 1}.ª")

A primeira componente explica 0,114 da variância, contra 0,6201 no `USArrests`. As sete primeiras juntas chegam só a 0,385, e as vinte primeiras, a 0,648: mesmo com vinte componentes, fica de fora 0,352, que é 1 menos 0,648.

In [ ]:
# Figura: A proporção da variância explicada por cada uma das 63 componentes do `NCI60`, com os genes padronizados (em cima) e a proporção acumulada (embaixo).
ks_nci = np.arange(1, 64)
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(3.8, 4.4),
    sharex=True)
ax1.plot(ks_nci, uteis, "o-",
    color="C0", markersize=3,
    linewidth=1)
ax2.plot(ks_nci, acum_nci, "o-",
    color="C0", markersize=3,
    linewidth=1)
ax1.set_ylabel("proporção")
ax2.set_ylabel("acumulada")
ax2.set_xlabel("componente")
ax1.set_ylim(0, 0.12)
ax2.set_ylim(0, 1.05)
ax2.set_xticks([1, 10, 20, 30,
    40, 50, 63])
ax2.set_xlim(-1, 65)
for ax in (ax1, ax2):
    ax.yaxis.set_major_formatter(
        virgula)
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> - **`ax.plot(..., markersize=3, linewidth=1)`** — pontos e linha mais finos, para caberem as 63 componentes.

Aqui o cotovelo é bem menos nítido. A maior queda é da primeira para a segunda componente; a partir da 4.ª, nenhuma queda entre vizinhas passa de 0,0055, e a curva desce devagar. Essa queda de 0,0055, da 6.ª para a 7.ª componente, é a última que se destaca, e termina na sétima: por isso se olham as sete primeiras. As quedas seguintes ficam todas abaixo dela, e o gráfico continua não decidindo onde fica o cotovelo.

A escolha fica objetiva quando as componentes servem a uma análise supervisionada. Na regressão por componentes principais da seção 11.6, o número $M$ de componentes é um parâmetro de ajuste, escolhido pelo erro de validação cruzada, como o $\lambda$ da ridge. Sem uma resposta, não há erro a medir, e a escolha continua sendo de quem analisa.

A regressão da seção 11.6 usa os escores no lugar das colunas originais, e o mesmo vale para outros métodos: uma classificação ou um agrupamento podem receber a tabela $n \times M$ dos $M$ primeiros escores, em vez da tabela $n \times p$ inteira. Quando a informação que interessa está concentrada nas primeiras componentes e as últimas carregam sobretudo ruído, trocar a tabela pelos primeiros escores pode deixar o resultado menos ruidoso. A seção 15.6 faz isso com o `NCI60`: agrupa as amostras pelos primeiros escores, em vez dos 6.830 genes.

De volta às perguntas do começo: com as colunas padronizadas, as duas primeiras componentes do `USArrests` guardam 0,8675 da variância, e 0,1325 fica fora do plano. E, padronizada, a tabela dá o mesmo resumo com as prisões por agressão contadas a cada 100 ou a cada 100.000 habitantes.

## k-Means

> **📌 Nota**
>
> Esta seção corresponde à seção 12.4.1 de James et al. (2023).

Cento e cinquenta pontos no plano parecem formar três nuvens, que se tocam nas bordas. Como dividi-los em $K$ grupos sem saber de que nuvem veio cada um? E, se o método começar de outro jeito, a divisão sai a mesma?

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("estilo-figuras.mplstyle")

A função `numero_br` escreve os números com a pontuação brasileira:

In [ ]:
numero_br = lambda v, casas=2: (
    f"{v:,.{casas}f}"
    .replace(",", "_")
    .replace(".", ",")
    .replace("_", ".")
    .replace("-", "−")
)

### O problema

Os pontos são simulados. Cada nuvem tem um centro, e cada ponto é o centro da sua nuvem mais um desvio sorteado, nas duas coordenadas:

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(15)
centros = np.array(
    [[0, 0], [4, 1], [1.5, 4]])
nuvem = np.repeat([0, 1, 2], 50)
desvio = rng.normal(size=(150, 2))
X = centros[nuvem] + desvio
print("forma:", X.shape)

> **🔧 Função**
>
> - **`np.random.default_rng(15)`** — um gerador de números aleatórios com a semente 15: toda execução sorteia os mesmos números. **`rng.normal(size=(150, 2))`** — 150 linhas e 2 colunas de sorteios da normal com média 0 e desvio padrão 1.
> - **`np.repeat([0, 1, 2], 50)`** — cada valor repetido 50 vezes seguidas: 50 zeros, 50 uns e 50 dois.
> - **`centros[nuvem]`** — uma linha de `centros` para cada elemento de `nuvem`: o centro da nuvem de cada ponto, 150 linhas ao todo.

Os 150 pontos, sem nenhuma indicação de nuvem:

In [ ]:
# Figura: Os 150 pontos simulados, sem grupo. Nos eixos, as duas coordenadas de `X`, na mesma escala.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(
    figsize=(3.4, 3.4))
ax.scatter(X[:, 0], X[:, 1],
    s=10, linewidths=0)
ax.set_aspect("equal")
ax.set_xlim(-3, 7)
ax.set_ylim(-3, 7)
ax.set_xticks([0, 3, 6])
ax.set_yticks([0, 3, 6])
plt.show()

O array `nuvem` só serve para construir os pontos; com dado real, ele não existe. O método recebe apenas `X`, as duas coordenadas, e `nuvem` fica guardado para conferir o resultado depois.

Dividir as $n$ observações em $K$ grupos é escolher conjuntos $C_1, \dots, C_K$ com os índices das observações de cada grupo, de modo que cada observação fique em exatamente um deles: se a observação $i$ está no grupo $k$, então $i \in C_k$.

Cada observação $i$ tem $p$ valores, $x_{i1}, \dots, x_{ip}$; aqui, $p = 2$, as duas colunas de `X`. Uma divisão é boa quando os pontos de cada grupo são parecidos entre si, e a medida mais comum de quão diferentes são dois pontos é a distância euclidiana ao quadrado.

> **🔷 Conceito**
>
> A **variação dentro do grupo** $C_k$ é a soma das distâncias euclidianas ao quadrado entre todos os pares ordenados $(i, i')$ de observações do grupo, dividida pelo número $|C_k|$ de observações dele. Os índices $i$ e $i'$ percorrem as observações do grupo, e cada par de pontos entra duas vezes, como $(i, i')$ e como $(i', i)$; os pares com $i = i'$ também entram, e valem 0:
>
> $$
> \begin{aligned}
> W(C_k) &= \frac{1}{|C_k|} \\
> &\quad \times \sum_{i,\,i' \in C_k}\sum_{j=1}^{p}
> (x_{ij} - x_{i'j})^2.
> \end{aligned}
> $$
>
> O ***k*-means**, um método de **agrupamento** (*clustering*), procura a divisão em $K$ grupos com a menor variação dentro dos grupos, somada nos $K$:
>
> $$
> \underset{C_1, \dots, C_K}{\text{minimizar}}
> \ \sum_{k=1}^{K} W(C_k).
> $$
>
> A soma $\sum_k W(C_k)$ é o **objetivo** do *k*-means.

> **🟩 Exemplo**
>
> Um grupo com os pontos $(0, 0)$ e $(3, 4)$, à distância 5 um do outro, tem quatro pares ordenados. Os dois de um ponto com ele mesmo valem 0, e o par do primeiro ponto com o segundo e o do segundo com o primeiro têm distância ao quadrado 25 cada: $W = (0 + 25 + 25 + 0)/2 = 25$. Um grupo de três pontos no mesmo lugar tem $W = 0$.

A conta do objetivo segue a definição: em cada grupo, as distâncias entre todos os pares, ao quadrado, somadas e divididas pelo tamanho do grupo. Ela vale para qualquer divisão. Aqui, ela se aplica a duas divisões: a das nuvens, que o método não conhece, e uma com rótulos 0, 1 e 2 sorteados ao acaso. Qual das duas deve ter o objetivo menor?

In [ ]:
from sklearn.metrics import (
    pairwise_distances)

def objetivo(X, rotulos):
    total = 0
    for k in np.unique(rotulos):
        A = X[rotulos == k]
        d = pairwise_distances(A)
        soma = (d**2).sum()
        total += soma / len(A)
    return total

ger = np.random.default_rng(15)
sorteio = ger.integers(0, 3, 150)
obj_nuv = objetivo(X, nuvem)
obj_sort = objetivo(X, sorteio)
print("objetivo das nuvens:",
    numero_br(obj_nuv))
print("objetivo do sorteio:",
    numero_br(obj_sort))
ct = pd.crosstab(nuvem, sorteio)
mistura = (ct > 0).all().all()
print("3 nuvens por grupo",
    "sorteado:", bool(mistura))

> **🔧 Função**
>
> - **`pairwise_distances(A)`**, do módulo `sklearn.metrics` — a tabela das distâncias euclidianas entre as linhas de `A`, duas a duas: o elemento $(i, i')$ é a distância entre a linha $i$ e a linha $i'$. Somar a tabela inteira é somar sobre os pares ordenados.
> - **`np.unique(rotulos)`** — os valores distintos de `rotulos`, em ordem. **`X[rotulos == k]`** — só as linhas de `X` em que o rótulo é `k`.
> - **`ger.integers(0, 3, 150)`** — 150 inteiros sorteados entre 0 e 2; o 3 fica de fora.
> - **`pd.crosstab(a, b)`** — conta quantas observações há em cada combinação de um valor de `a`, nas linhas, com um valor de `b`, nas colunas. **`(ct > 0).all().all()`** — `True` quando nenhuma contagem é zero: o primeiro `all` confere cada coluna, e o segundo junta as colunas.

A divisão das nuvens tem objetivo 614,88, e a sorteada, 2.228,95. Com rótulos ao acaso, cada grupo sorteado tem pontos das três nuvens, e os pares de pontos distantes pesam no objetivo.

Para achar a divisão de menor objetivo, bastaria calcular o objetivo de todas? Cada um dos 150 pontos pode ir para qualquer dos 3 grupos:

In [ ]:
total = 3**150
print("algarismos de 3^150:",
    len(str(total)))
sem_nome = total // 6
print("dividido por 6:",
    len(str(sem_nome)))

São $3^{150}$ maneiras de pôr os rótulos, um número de 72 algarismos. Duas maneiras que só trocam os nomes dos grupos dão a mesma divisão; descontadas essas trocas, sobra perto de $3^{150}/6$, ainda um número de 71 algarismos. Com $n$ observações e $K$ grupos, o número de divisões cresce como $K^n$, e só é pequeno quando $K$ e $n$ são minúsculos. Nenhum computador as percorre todas.

O `KMeans`, do `scikit-learn`, procura uma divisão de objetivo pequeno sem percorrer todas. Por padrão, ele começa de centroides escolhidos entre os próprios pontos, e não de rótulos sorteados como no algoritmo descrito adiante; as voltas seguintes são as mesmas. Com $K = 2$, $3$ e $4$, o objetivo e os tamanhos dos grupos:

In [ ]:
from sklearn.cluster import KMeans

modelos = {}
objetivos = []
print("K: objetivo · tamanhos")
for K in [2, 3, 4]:
    km = KMeans(n_clusters=K,
        n_init=20,
        random_state=15)
    km.fit(X)
    modelos[K] = km
    obj = objetivo(X, km.labels_)
    objetivos.append(obj)
    tam = np.sort(
        np.bincount(km.labels_))
    print(f"{K}:", numero_br(obj),
        "·", *tam)
quedas = np.diff(objetivos)
cai = np.all(quedas < 0)
print("cai a cada K:", bool(cai))

> **🔧 Função**
>
> - **`KMeans(n_clusters=K, n_init=20, random_state=15)`**, do módulo `sklearn.cluster` — o *k*-means com `K` grupos. O algoritmo depende de um começo sorteado: `n_init=20` o roda a partir de 20 começos e fica com o de menor objetivo, e `random_state` fixa a semente dos sorteios.
> - **`km.fit(X)`** — divide as linhas de `X` em grupos. **`km.labels_`** — o rótulo do grupo de cada linha, um inteiro de 0 a `K − 1`.
> - **`np.bincount(arr)`** — quantas vezes aparece cada inteiro 0, 1, 2, … de `arr`: aqui, o tamanho de cada grupo. **`np.sort(arr)`** — os valores em ordem crescente. Os rótulos não têm ordem, e os tamanhos, ordenados, não dependem de qual grupo recebeu qual rótulo.

O objetivo cai a cada grupo a mais: de 1.342,12 com dois grupos para 576,51 com três e 474,15 com quatro. Na melhor divisão, isso vem da construção: com um grupo a mais, dá para partir um grupo em dois sem aumentar o objetivo, como a parte sobre o algoritmo mostra. Por isso a melhor divisão com $K + 1$ grupos nunca tem objetivo maior que a melhor com $K$. O objetivo não serve, então, para escolher $K$, e a seção 15.6 volta a essa escolha.

As três divisões, lado a lado:

In [ ]:
# Figura: A divisão do *k*-means com $K = 2$, $3$ e $4$. Cada cor é um grupo, e cada X grande, o centroide dele, a média dos pontos do grupo. As cores não têm ordem: são só nomes para os grupos. Nos eixos, as duas coordenadas de `X`.
cores = np.array(
    ["C1", "C2", "C3", "0.45"])

def painel(ax, rotulos=None,
        centr=None, tam=110):
    cor = ("C0" if rotulos is None
        else cores[rotulos])
    ax.scatter(X[:, 0], X[:, 1],
        s=7, c=cor, linewidths=0)
    if centr is not None:
        k = len(centr)
        ax.scatter(centr[:, 0],
            centr[:, 1], s=tam,
            marker="X",
            c=cores[:k],
            edgecolors="white")
    ax.set_aspect("equal")
    ax.set_xlim(-3, 7)
    ax.set_ylim(-3, 7)
    ax.set_xticks([0, 3, 6])
    ax.set_yticks([0, 3, 6])

fig, eixos = plt.subplots(
    1, 3, figsize=(3.8, 1.8),
    sharey=True)
for ax, K in zip(eixos,
        [2, 3, 4]):
    km = modelos[K]
    painel(ax, km.labels_,
        km.cluster_centers_)
    ax.set_title(f"K = {K}")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> - **`ax.scatter(..., c=cores[rotulos])`** — uma cor por ponto: `cores[rotulos]` troca cada rótulo pela cor da posição dele na lista. **`marker="X"`** desenha um X cheio, e **`edgecolors="white"`**, o contorno branco que o separa dos pontos.
> - **`km.cluster_centers_`** — o centroide de cada grupo, uma linha por grupo; `centr[:, 0]` e `centr[:, 1]` são as duas coordenadas deles.
> - **`sharey=True`** — os três gráficos dividem o eixo vertical.

Com $K = 3$, os grupos batem com as nuvens? Os rótulos do `KMeans` são nomes arbitrários, e o rótulo 0 dele não tem nada a ver com a nuvem 0. A comparação se faz numa tabela cruzada, com as nuvens nas linhas e os grupos nas colunas:

In [ ]:
km3 = modelos[3]
tabela = pd.crosstab(nuvem,
    km3.labels_,
    rownames=["nuvem"],
    colnames=["grupo"])
print(tabela)
maior = tabela.max(axis=1)
print("maioria em cada nuvem:",
    bool((maior > 25).all()))
dono = tabela.idxmax(axis=1)
print("grupos da maioria",
    "distintos:",
    dono.nunique() == 3)
print("fora da maioria da nuvem:",
    150 - maior.sum())
obj3 = objetivo(X, km3.labels_)
menor = obj3 < obj_nuv
print("menor que o das nuvens:",
    bool(menor))

> **🔧 Função**
>
> - **`pd.crosstab(..., rownames, colnames)`** — `rownames` e `colnames` dão o nome das linhas e das colunas da tabela cruzada.
> - **`tabela.max(axis=1)`** — o maior valor de cada linha. **`tabela.idxmax(axis=1)`** — a coluna em que está o maior valor de cada linha: o grupo que fica com a maior parte de cada nuvem. **`.nunique()`** — quantos valores distintos há.

Cada nuvem tem um grupo com a maioria dos seus pontos, os três grupos são diferentes, e 6 dos 150 pontos ficam fora do grupo da maioria da sua nuvem. O *k*-means errou esses 6? Para cada um, a distância ao quadrado até o centroide do grupo em que ficou e até o centroide do grupo da maioria da sua nuvem:

In [ ]:
C = km3.cluster_centers_
dono_pt = dono.to_numpy()[nuvem]
seu = C[km3.labels_]
maj = C[dono_pt]
d_seu = ((X - seu)**2).sum(axis=1)
d_maj = ((X - maj)**2).sum(axis=1)
fora_m = km3.labels_ != dono_pt
perto = (d_seu < d_maj)[fora_m]
n_fora = fora_m.sum()
print(f"os {n_fora} fora",
    "da maioria:")
print("  perto do próprio",
    "centroide:",
    bool(perto.all()))

> **🔧 Função**
>
> - **`C[km3.labels_]`** — o centroide do grupo de cada ponto, como `centros[nuvem]` fez com o centro da nuvem; **`dono.to_numpy()[nuvem]`**, do mesmo jeito, dá a cada ponto o grupo da maioria da sua nuvem.
> - **`(d_seu < d_maj)[fora_m]`** — compara, ponto a ponto, as duas distâncias, e o `[fora_m]` fica só com os 6 pontos fora da maioria.

Não: o objetivo da divisão do *k*-means é menor que o da divisão das nuvens, e cada um desses pontos está mais perto do centroide do grupo em que ficou do que do centroide do grupo da maioria da sua nuvem.

### O algoritmo

Como achar uma divisão boa sem percorrer todas? O algoritmo é curto:

1. Sorteie para cada observação um rótulo entre os $K$ possíveis. É a divisão inicial.
2. Repita até a divisão parar de mudar:
   a. Para cada grupo, calcule o **centroide**, o ponto cujas coordenadas são as médias das $p$ colunas nas observações do grupo.
   b. Mude cada observação para o grupo do centroide mais próximo, pela distância euclidiana.

O nome vem do passo 2(a): cada grupo é resumido pela média, *mean*, dos seus pontos. O passo 2(a), à mão, sobre os rótulos já sorteados em `sorteio` (o passo 1):

In [ ]:
tabela_X = pd.DataFrame(X)
cent1 = (tabela_X
    .groupby(sorteio).mean()
    .to_numpy())
d_cent = pairwise_distances(cent1)
print("centroides, maior",
    "distância:",
    numero_br(d_cent.max()))
d_nuv = pairwise_distances(
    centros)
pos = d_nuv[d_nuv > 0]
print("nuvens, menor distância:",
    numero_br(pos.min()))

> **🔧 Função**
>
> - **`df.groupby(rotulos).mean()`** — separa as linhas da tabela pelo valor de `rotulos` e tira a média de cada coluna dentro de cada grupo: uma linha por grupo, com as médias.

Como cada grupo sorteado tem pontos das três nuvens, as médias saem quase iguais: a maior distância entre dois centroides é 0,42, e a menor entre os centros de duas nuvens, 3,91. Agora o passo 2(b):

In [ ]:
from sklearn.metrics import (
    pairwise_distances_argmin)

rot1 = pairwise_distances_argmin(
    X, cent1)
mudaram = (rot1 != sorteio).sum()
print("mudaram de grupo:",
    mudaram)
print("objetivo:",
    numero_br(obj_sort), "→",
    numero_br(objetivo(X, rot1)))

> **🔧 Função**
>
> - **`pairwise_distances_argmin(X, cent1)`** — para cada linha de `X`, a posição da linha de `cent1` mais próxima dela: o rótulo do centroide mais perto de cada ponto.
> - **`(rot1 != sorteio).sum()`** — quantos pontos têm o rótulo novo diferente do antigo; cada `True` conta 1.

Com os centroides quase juntos, 89 dos 150 pontos trocam de grupo, e o objetivo cai de 2.228,95 para 1.118,51. Mais uma volta, com os centroides dos grupos novos:

In [ ]:
cent2 = (tabela_X
    .groupby(rot1).mean()
    .to_numpy())
rot2 = pairwise_distances_argmin(
    X, cent2)
mudaram = (rot2 != rot1).sum()
print("mudaram de grupo:",
    mudaram)
print("objetivo:",
    numero_br(objetivo(X, rot2)))

Na segunda volta mudam só 18 pontos, e o objetivo cai para 976,23. Daqui em diante, o `KMeans` repete as voltas até a divisão parar de mudar. Com `init=cent1`, ele parte dos mesmos centroides do passo 2(a):

In [ ]:
km_mao = KMeans(n_clusters=3,
    init=cent1, n_init=1)
km_mao.fit(X)
print("voltas:", km_mao.n_iter_)
rot_f = km_mao.labels_
final = objetivo(X, rot_f)
print("objetivo final:",
    numero_br(final))
passos = [obj_sort,
    objetivo(X, rot1),
    objetivo(X, rot2), final]
subiu = np.any(
    np.diff(passos) > 0)
print("subiu em algum passo:",
    bool(subiu))

> **🔧 Função**
>
> - **`KMeans(n_clusters=3, init=cent1, n_init=1)`** — o *k*-means a partir dos centroides dados em `init`, em vez de sorteá-los; com um começo fixo, só há um começo a rodar, e `n_init=1` diz isso.
> - **`km.n_iter_`** — quantas voltas do passo 2 o algoritmo deu até parar.
> - **`np.any(arr)`** — `True` quando ao menos um elemento é `True`.

São 7 voltas ao todo, e o objetivo termina em 576,85. Nas quatro divisões, a sorteada, as duas obtidas nas voltas feitas à mão e a final, o objetivo não subiu em nenhum passo. A figura mostra as mesmas etapas:

In [ ]:
# Figura: O *k*-means com $K = 3$, a partir de rótulos sorteados. Passo 1: os rótulos sorteados, com as três cores misturadas. Passo 2(a): os centroides desses grupos, os X, tão juntos no meio que quase se sobrepõem; os tamanhos diferentes deixam ver os três. Passo 2(b): cada ponto recebe a cor do centroide mais próximo. Segunda volta: os centroides dos grupos novos, e de novo cada ponto com a cor do mais próximo. Final: a divisão depois de 7 voltas, com os centroides dela. Nos eixos, as duas coordenadas de `X`.
titulos = ["passo 1",
    "passo 2(a)", "passo 2(b)",
    "volta 2, 2(a)",
    "volta 2, 2(b)", "final"]
rots = [sorteio, sorteio, rot1,
    rot1, rot2, km_mao.labels_]
cents = [None, cent1, cent1,
    cent2, cent2,
    km_mao.cluster_centers_]
juntos = [240, 150, 70]
tams = [110, juntos, juntos,
    110, 110, 110]
fig, eixos = plt.subplots(
    3, 2, figsize=(3.8, 5.6),
    sharex=True, sharey=True)
for i in range(6):
    ax = eixos.flat[i]
    painel(ax, rots[i], cents[i],
        tams[i])
    ax.set_title(titulos[i],
        fontsize=10)
plt.tight_layout()
plt.show()

Por que o objetivo nunca sobe? A variação dentro do grupo, definida pelos pares, tem uma segunda forma, pela distância de cada ponto ao centroide $\bar{x}_k$ do grupo:

$$
\begin{aligned}
&\frac{1}{|C_k|} \\
&\quad \times \sum_{i,\,i' \in C_k}\ \sum_{j=1}^{p}
(x_{ij} - x_{i'j})^2 \\
&\quad = 2 \sum_{i \in C_k}\ \sum_{j=1}^{p}
(x_{ij} - \bar{x}_{kj})^2,
\end{aligned}
$$

em que $\bar{x}_{kj}$ é a média da coluna $j$ no grupo $k$. O 2 vem dos pares ordenados: cada par de pontos entra duas vezes na soma da esquerda. A soma da direita, sem o 2 e somada nos $K$ grupos, é o que o `KMeans` guarda no atributo `inertia_`, a **inércia**. O objetivo é, então, o dobro da inércia. Na divisão final:

In [ ]:
inercia = km_mao.inertia_
print("inércia:",
    numero_br(inercia))
print("objetivo pelos pares:",
    numero_br(final))
dobro = np.isclose(final,
    2 * inercia)
print("objetivo = 2 × inércia:",
    bool(dobro))

> **🔧 Função**
>
> - **`km.inertia_`** — a inércia: a soma, em todos os pontos, da distância ao quadrado até o centroide do próprio grupo.

A inércia é 288,42, e o objetivo, 576,85, o dobro dela a menos do arredondamento da última casa.

Para ver por que nenhum passo aumenta o objetivo, chame de $S$ a soma, em todos os pontos, da distância ao quadrado de cada ponto ao centroide que ele está usando, seja qual for esse centroide. Quando os centroides são as médias dos grupos, $S$ é a inércia, e o objetivo vale $2S$. Três valores de $S$: com o sorteio e os centroides dele, depois do passo 2(b) da primeira volta e depois do passo 2(a) da segunda:

In [ ]:
def S(rotulos, centr):
    dif = X - centr[rotulos]
    return (dif**2).sum()

s_a = S(sorteio, cent1)
s_b = S(rot1, cent1)
s_c = S(rot1, cent2)
print("S, sorteio e cent1:",
    numero_br(s_a))
print("S, rot1 e cent1:",
    numero_br(s_b))
print("S, rot1 e cent2:",
    numero_br(s_c))
print("S só desceu:",
    bool(s_a > s_b > s_c))

No passo 2(b), os centroides ficam fixos. Cada ponto só troca o seu centroide por um mais perto, e $S$ cai de 1.114,47 para 996,80.

No passo 2(a), a divisão fica fixa, e cada centroide vira a média do seu grupo. A média é o ponto que torna mínima a soma das distâncias ao quadrado até os pontos do grupo. Com os números 0, 1 e 5, de média 2, essa soma é $4 + 1 + 9 = 14$ até a média, $1 + 0 + 16 = 17$ até o 1 e $9 + 4 + 4 = 17$ até o 3. Como cada centroide novo é a média do seu grupo, $S$ cai de novo, para 559,25, a inércia da divisão nova.

A cadeia vale em toda volta: a inércia nova não passa de $S$ com os centroides velhos, que não passa da inércia velha. Como o objetivo é o dobro da inércia, ele também nunca sobe. É o mesmo fato que explica por que o objetivo cai com $K$: usar a média de cada parte de um grupo nunca soma mais que usar a média do grupo inteiro.

Quando nenhum ponto muda, a divisão está parada num **ótimo local**: nenhum passo do algoritmo a melhora, mas outra divisão ainda pode ter objetivo menor.

### Ótimos locais

O algoritmo que partiu de `sorteio` parou com objetivo 576,85, e o `KMeans` com 20 começos, na tabela de $K = 2$, $3$ e $4$, chegou a 576,51, um pouco menos. O resultado depende do sorteio do passo 1. Com seis sorteios diferentes, de sementes 0 a 5, quantos resultados diferentes aparecem?

In [ ]:
seis = []
print("semente: objetivo",
    "· tamanhos")
for s in range(6):
    g = np.random.default_rng(s)
    rot = g.integers(0, 3, 150)
    cent = (tabela_X.groupby(rot)
        .mean().to_numpy())
    km = KMeans(n_clusters=3,
        init=cent, n_init=1)
    km.fit(X)
    seis.append(km)
    obj = 2 * km.inertia_
    tam = np.sort(
        np.bincount(km.labels_))
    print(f"{s}:", numero_br(obj),
        "·", *tam)
objs = [2 * km.inertia_
    for km in seis]
distintos = np.unique(
    np.round(objs, 2))
print("objetivos distintos:",
    len(distintos))
t2 = pd.crosstab(nuvem,
    seis[2].labels_).to_numpy()
por_col = np.sort(t2, axis=0)
seg = np.sort(por_col[-2])
print("semente 2, pontos da 2.ª")
print("  nuvem de cada grupo:",
    *seg)

> **🔧 Função**
>
> - **`np.round(arr, 2)`** — arredonda cada valor a 2 casas, para que objetivos que só diferem depois da segunda casa contem como um só.
> - **`np.sort(t2, axis=0)`** — ordena cada coluna da tabela cruzada, a contagem de cada nuvem num grupo; a penúltima linha, `[-2]`, é a segunda maior contagem de cada grupo.

O número impresso é o objetivo, o dobro da inércia. Os seis começos param em três ótimos locais diferentes. Dois deles, de objetivo 576,51 e 576,85, quase se confundem, com tamanhos de grupo quase iguais.

O terceiro, o da semente 2, de objetivo 1.158,11, é outra divisão, com um grupo de 64 pontos e dois menores, e cada grupo tem ao menos 18 pontos de uma segunda nuvem:

In [ ]:
# Figura: O *k*-means com $K = 3$ a partir de seis sorteios diferentes de rótulos. Acima de cada painel, a semente do sorteio e o objetivo da divisão final, o dobro da inércia. As cores são só nomes para os grupos e mudam de um painel para outro sem significar nada. Nos eixos, as duas coordenadas de `X`.
fig, eixos = plt.subplots(
    3, 2, figsize=(3.8, 5.6),
    sharex=True, sharey=True)
for s in range(6):
    ax = eixos.flat[s]
    km = seis[s]
    painel(ax, km.labels_,
        km.cluster_centers_)
    obj = 2 * km.inertia_
    tit = f"semente {s}: "
    tit += numero_br(obj)
    ax.set_title(tit, fontsize=10)
plt.tight_layout()
plt.show()

Como o algoritmo para num ótimo local, a saída é rodá-lo a partir de vários começos e ficar com o resultado de menor objetivo. É o que faz o argumento `n_init`. Quantos começos o `KMeans` roda se ninguém disser?

In [ ]:
padrao = KMeans(n_clusters=3)
print("n_init padrão:",
    padrao.n_init)
inerc_um = [KMeans(n_clusters=3,
    n_init=1, random_state=s)
    .fit(X).inertia_
    for s in range(20)]
objs_um = np.unique(np.round(
    2 * np.array(inerc_um), 2))
print("um começo, 20 sementes:")
print("  objetivos distintos:",
    len(objs_um))
km50 = KMeans(n_clusters=3,
    n_init=50, random_state=15)
km50.fit(X)
obj50 = 2 * km50.inertia_
print("50 começos:",
    numero_br(obj50))
igual = np.isclose(obj50,
    min(objs))
print("igual ao menor dos seis:",
    bool(igual))
t50 = pd.crosstab(nuvem,
    km50.labels_)
dono50 = t50.idxmax(axis=1)
print("nuvens em grupos",
    "distintos:",
    dono50.nunique() == 3)
print("fora da maioria:",
    150 - t50.max(axis=1).sum())

> **🔧 Função**
>
> - **`n_init="auto"`** — o padrão do `KMeans`: com a inicialização padrão, `init="k-means++"`, roda **um** começo só. O `k-means++` não sorteia rótulos: sorteia os centroides iniciais entre os próprios pontos, dando mais chance aos pontos distantes dos centroides já escolhidos.
> - **`KMeans(...).fit(X).inertia_`** — o `fit` devolve o próprio modelo, e dá para ler a inércia na mesma linha.

Mesmo com o `k-means++`, um começo por semente dá 2 objetivos diferentes em 20 sementes. Com 50 começos, o objetivo é 576,51, o menor dos seis sorteios.

Outra divisão ainda pode ter objetivo menor. Passe sempre `n_init` e `random_state` ao `KMeans`: o primeiro dá vários começos, e o segundo faz o resultado se repetir a cada execução.

> **⚠️ Atenção**
>
> O *k*-means recebe $K$ de fora e devolve $K$ grupos mesmo quando o dado não tem grupo nenhum. A distância euclidiana soma as colunas como vêm, e uma coluna de valores grandes manda mais que as outras. E os grupos que ele forma tendem a ser compactos, em torno de um centro. A seção 15.6 trata da escala das colunas, da escolha de $K$ e de como conferir se os grupos se sustentam.

A divisão dos 150 pontos em $K$ grupos é definida pelo objetivo, e o *k*-means procura uma boa alternando médias e reatribuições. O resultado não é sempre o mesmo: depende do começo, e seis sorteios deram três ótimos locais. Com 50 começos, fica o de menor objetivo, que aqui põe cada nuvem num grupo diferente, com 6 dos 150 pontos fora do grupo da maioria da sua nuvem.

## Clustering Hierárquico

> **📌 Nota**
>
> Esta seção corresponde à seção 12.4.2 de James et al. (2023).

Para dividir observações com o *k*-means, é preciso dizer antes quantos grupos se quer. Dá para agrupar sem escolher esse número e ver, de uma vez, as divisões em 1, 2, 3, …, até $n$ grupos? E, se der, qual delas escolher?

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("estilo-figuras.mplstyle")

A função `numero_br` escreve os números com a pontuação brasileira:

In [ ]:
numero_br = lambda v, casas=2: (
    f"{v:,.{casas}f}"
    .replace(",", "_")
    .replace(".", ",")
    .replace("_", ".")
    .replace("-", "−")
)

### Lendo um dendrograma

Os dados são 45 pontos simulados no plano, 15 em torno de cada um de três centros:

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(15)
centros = np.array(
    [[-4, 1], [0, -1], [1, 3]])
classe = np.repeat([0, 1, 2], 15)
desvio = rng.normal(size=(45, 2))
X = centros[classe] + desvio
print("forma:", X.shape)

Como no *k*-means, o array `classe` só constrói os pontos: o método recebe apenas `X`, e `classe` fica guardado para conferir o resultado.

In [ ]:
# Figura: Os 45 pontos simulados, sem grupo. Nos eixos, as duas coordenadas de `X`, na mesma escala.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(
    figsize=(3.4, 2.6))
ax.scatter(X[:, 0], X[:, 1],
    s=12, linewidths=0)
ax.set_aspect("equal")
ax.set_xticks([-6, -3, 0, 3])
ax.set_yticks([-3, 0, 3, 6])
plt.show()

O **agrupamento hierárquico** (*hierarchical clustering*) aglomerativo começa com cada observação num grupo só dela e, a cada passo, funde os dois grupos mais parecidos, até sobrar um grupo com todas as observações. A **dissimilaridade** mede quanto dois grupos diferem, e os mais parecidos são os de menor dissimilaridade. Entre duas observações, a dissimilaridade aqui é a distância euclidiana. O registro das fusões é uma árvore, o dendrograma. O `scipy` constrói as fusões com a função `linkage`:

In [ ]:
from scipy.cluster.hierarchy \
    import linkage

Z = linkage(X, "complete")
print("fusões:", len(Z))
alturas = Z[:, 2]
sobem = np.all(
    np.diff(alturas) >= 0)
print("alturas só sobem:",
    bool(sobem))
print("as 4 mais altas:")
print(" ", *[numero_br(a)
    for a in alturas[-4:]])

> **🔧 Função**
>
> - **`linkage(X, "complete")`**, do módulo `scipy.cluster.hierarchy` — o agrupamento hierárquico das linhas de `X`, com a distância euclidiana. O `"complete"` escolhe a regra que mede a dissimilaridade entre dois grupos, a ligação, definida adiante, em "A ligação". Devolve a **matriz de ligação**, uma linha por fusão, na ordem em que acontecem: as duas primeiras colunas dizem quais grupos se fundem, a terceira é a altura da fusão, e a quarta, quantas observações tem o grupo novo.
> - **`Z[:, 2]`** — a terceira coluna, a das alturas.

Cada fusão junta dois grupos em um, e de 45 grupos até 1 são 44 fusões. A altura de uma fusão é a dissimilaridade entre os dois grupos que se juntam, e aqui ela só sobe de uma fusão para a seguinte: as quatro últimas são as mais altas. As duas últimas, a 7,89 e a 9,22, ficam bem acima das duas anteriores, a 3,93 e a 4,38.

Na figura, a árvore aparece três vezes; a segunda e a terceira têm uma linha tracejada, em 8,5 e em 6. Quantos galhos cada linha atravessa?

In [ ]:
# Figura: O dendrograma dos 45 pontos, com a ligação completa e a distância euclidiana. Em cima, sem corte. No meio, o corte na altura 8,5, a linha tracejada, e embaixo, o corte na altura 6; em cada um, os galhos abaixo do corte têm a cor do seu grupo. As cores não entram no agrupamento: só nomeiam os grupos.
from scipy.cluster.hierarchy \
    import dendrogram

cortes = [None, 8.5, 6]
# argumentos das três chamadas;
# o ** os passa com seus nomes
opcoes = dict(no_labels=True,
    above_threshold_color="0.45")
fig, eixos = plt.subplots(
    3, 1, figsize=(3.8, 6.0),
    sharex=True)
for ax, h in zip(eixos, cortes):
    limiar = 0 if h is None else h
    dendrogram(Z, ax=ax,
        color_threshold=limiar,
        **opcoes)
    if h is not None:
        ax.axhline(h, ls="--",
            color="0.45", lw=1)
    ax.set_ylabel("altura")
    ax.set_yticks([0, 5, 10])
    ax.grid(axis="x",
        visible=False)
eixos[0].set_title("sem corte",
    fontsize=10)
eixos[1].set_title("corte em 8,5",
    fontsize=10)
eixos[2].set_title("corte em 6",
    fontsize=10)
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> - **`dendrogram(Z, ax=ax, ...)`** — desenha a árvore da matriz de ligação `Z` no gráfico `ax`. Os galhos inteiramente abaixo da altura `color_threshold` ganham a cor do seu grupo, e os demais, a cor `above_threshold_color`; com `color_threshold=0`, tudo fica nessa cor. **`no_labels=True`** esconde os números das folhas, que com 45 observações se amontoariam.
> - **`ax.axhline(h, ...)`** — uma linha horizontal na altura `h`, de uma ponta à outra do gráfico; `ls="--"` a faz tracejada.
> - **`ax.grid(axis="x", visible=False)`** — tira as linhas verticais da malha, que no dendrograma não marcam nada.

> **🔷 Conceito**
>
> No **dendrograma**, cada folha é uma observação, e cada fusão, um traço horizontal que liga dois galhos, na altura dela. Duas observações que se juntam num galho baixo são parecidas; duas que só se juntam perto do topo podem ser muito diferentes.
>
> Um **corte** horizontal na altura $h$ divide as observações em grupos: os galhos que o corte atravessa. Quanto mais baixo o corte, mais grupos, de 1 (acima da última fusão) até $n$ (abaixo da primeira).

#### Cortando a árvore

Por que cortar em 8,5 e em 6 (2 e 3 grupos)? Qualquer corte entre 7,89 e 9,22 atravessa os mesmos 2 galhos, e qualquer corte entre 4,38 e 7,89, os mesmos 3. O 8,5 e o 6 caem nesses dois intervalos. O de 3 grupos é largo porque a altura salta de 4,38 para 7,89, uma diferença de 3,51. Um salto grande entre fusões seguidas marca um corte natural: abaixo dele, os grupos se formam por fusões baixas, e acima só se juntam grupos bem diferentes.

O corte faz o papel do $K$ do *k*-means: decide quantos grupos saem. A diferença é que o corte vem depois, sobre uma árvore já construída, e mudar de ideia não pede rodar nada de novo. O `fcluster` faz o corte e devolve o grupo de cada observação. Os grupos batem com as três classes da simulação?

In [ ]:
from scipy.cluster.hierarchy \
    import fcluster

g2 = fcluster(Z, 8.5,
    criterion="distance")
g3 = fcluster(Z, 6,
    criterion="distance")
print(pd.crosstab(classe, g2,
    rownames=["classe"],
    colnames=["corte 8,5"]))
print(pd.crosstab(classe, g3,
    rownames=["classe"],
    colnames=["corte 6"]))
ct = pd.crosstab(g3, g2)
dentro = (ct > 0).sum(axis=1)
print("grupos do 6 dentro do",
    "8,5:",
    bool((dentro == 1).all()))

> **🔧 Função**
>
> - **`fcluster(Z, h, criterion="distance")`** — corta a árvore na altura `h` e devolve, para cada observação, o número do grupo em que ela ficou, a partir de 1. Os números são só nomes, como os rótulos do *k*-means.
> - **`(ct > 0).sum(axis=1)`** — em cada linha da tabela cruzada, em quantas colunas há alguma observação: aqui, em quantos grupos do corte de cima caem as observações de cada grupo do corte de baixo.

O corte em 8,5 dá dois grupos: um com toda a classe 2, a nuvem de cima, e outro, de 30, com as classes 0 e 1 juntas. O corte em 6 separa as três classes, uma por grupo. Por que a nuvem de cima se separa antes das outras duas? A resposta depende de como se mede a dissimilaridade entre dois grupos, e volta na parte "A ligação".

Os grupos do corte mais baixo cabem, cada um, dentro de um grupo do corte mais alto: o corte em 6 só partiu em dois o grupo de 30. Esse encaixe vale para quaisquer dois cortes da mesma árvore, e é dele que vem o nome **hierárquico**. O encaixe é também um limite.

Imagine pessoas divididas igualmente entre homens e mulheres, e entre brasileiros, japoneses e franceses. A melhor divisão em dois grupos pode ser por sexo, e a melhor em três, por país. Os três grupos de país não cabem dentro dos dois de sexo, e nenhum corte de uma árvore dá as duas divisões. Num caso assim, a árvore acerta no máximo uma das duas, e o *k*-means, rodado uma vez com $K = 2$ e outra com $K = 3$, não tem essa amarra.

O `scikit-learn` tem o mesmo método, no `AgglomerativeClustering`. Ele não desenha a árvore, mas devolve os grupos de um corte, dado o número de grupos:

In [ ]:
from sklearn.cluster import (
    AgglomerativeClustering)
from sklearn.metrics import (
    adjusted_rand_score)

ag = AgglomerativeClustering(
    n_clusters=3,
    linkage="complete")
ag.fit(X)
ari = adjusted_rand_score(
    ag.labels_, g3)
print("mesma divisão do corte 6:",
    bool(ari == 1))

> **🔧 Função**
>
> - **`AgglomerativeClustering(n_clusters=3, linkage="complete")`**, do módulo `sklearn.cluster` — o agrupamento hierárquico, cortado em `n_clusters` grupos. O argumento `linkage` precisa ser dado para igualar o `scipy`: o padrão, `"ward"`, é outra regra.
> - **`ag.labels_`** — o grupo de cada observação, a partir de 0.
> - **`adjusted_rand_score(a, b)`**, o índice de Rand ajustado, do módulo `sklearn.metrics` — compara duas divisões das mesmas observações, sem olhar os nomes dos grupos: vale 1 quando as duas divisões coincidem, e fica perto de 0 quando não têm relação uma com a outra.

Os nomes dos grupos são outros, 0, 1 e 2, em vez de 1, 2 e 3, mas a divisão é a mesma.

### Proximidade na horizontal

Numa árvore pequena dá para ver as fusões uma a uma. São nove pontos sorteados no plano, numerados de 1 a 9:

In [ ]:
ger = np.random.default_rng(15)
pontos = ger.uniform(
    -1.5, 1, size=(9, 2))
Z9 = linkage(pontos, "complete")
arvore = dendrogram(Z9,
    no_plot=True)
folhas = np.array(
    arvore["leaves"]) + 1
print("folhas, da esquerda à",
    "direita:")
print(" ", *folhas)

> **🔧 Função**
>
> - **`ger.uniform(-1.5, 1, size=(9, 2))`** — 9 linhas e 2 colunas de sorteios uniformes entre −1,5 e 1.
> - **`dendrogram(Z9, no_plot=True)`** — calcula o desenho sem desenhar; a chave `"leaves"` traz as observações na ordem das folhas, da esquerda para a direita, contadas a partir de 0. O `+ 1` as numera de 1 a 9.

As folhas 4 e 8 ficam lado a lado. Isso quer dizer que os pontos 4 e 8 são parecidos? Na árvore, o que diz quão parecidos são dois pontos é a altura em que eles se juntam pela primeira vez. A raiz, a última fusão, junta dois galhos; em qual deles fica cada ponto?

In [ ]:
raiz = Z9[-1, 2]
print("altura da raiz:",
    numero_br(raiz))
lado = fcluster(Z9, 2,
    criterion="maxclust")
for k in [4, 8]:
    mesmo = lado == lado[k - 1]
    quem = np.flatnonzero(mesmo)
    print(f"com o {k}:",
        *quem + 1)
fusoes = len(Z9)
print("fusões:", fusoes)
print("desenhos da mesma árvore:",
    2 ** fusoes)

> **🔧 Função**
>
> - **`fcluster(Z9, 2, criterion="maxclust")`** — corta a árvore na altura que dá 2 grupos, em vez de numa altura dada: os dois galhos que a raiz junta.
> - **`np.flatnonzero(arr)`** — as posições em que `arr` é `True`; o `+ 1` as numera de 1 a 9.

O 4 e o 8 ficam em galhos diferentes da raiz e só se juntam nela, na altura 2,34. Pela árvore, o 4 é tão diferente do 8 quanto do 1, do 5, do 6 e do 7, que estão no mesmo galho que o 8; o 8 fica ao lado dele no desenho por acaso da ordem.

Em cada uma das 8 fusões, os dois galhos podem trocar de lado sem mudar a árvore, o que dá $2^{8} = 256$ desenhos da mesma árvore. Trocar de lado só os dois galhos da raiz já põe o 8 numa ponta do desenho e o 4 na outra. Quão parecidos são dois pontos se lê no eixo vertical, na altura em que eles se juntam; a ordem das folhas no eixo horizontal é só uma das 256.

In [ ]:
# Figura: Em cima, o dendrograma dos nove pontos, com a ligação completa e a distância euclidiana, e o número de cada ponto embaixo da sua folha. Embaixo, os nove pontos no plano, com os mesmos números. As folhas 4 e 8, lado a lado no dendrograma, estão em laranja nos dois gráficos.
cor_pt = np.array(["C0"] * 9)
cor_pt[[3, 7]] = "C1"
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(3.8, 5.2),
    height_ratios=[1, 1.3])
dendrogram(Z9, ax=ax1,
    labels=np.arange(1, 10),
    color_threshold=0,
    above_threshold_color="0.45",
    leaf_font_size=10)
for rot in ax1.get_xticklabels():
    k = int(rot.get_text())
    rot.set_color(cor_pt[k - 1])
ax1.set_ylabel("altura")
ax1.grid(axis="x", visible=False)
ax2.scatter(pontos[:, 0],
    pontos[:, 1], s=30,
    c=cor_pt, linewidths=0)
desl = "offset points"
# número acima e à direita do
# ponto; o do 1 vai para baixo,
# para não encostar no 6
lugar = [(5, 3)] * 9
lugar[0] = (-4, -13)
for k in range(9):
    x, y = pontos[k]
    ax2.annotate(str(k + 1),
        (x, y), xytext=lugar[k],
        textcoords=desl,
        fontsize=10,
        color=cor_pt[k])
ax2.set_aspect("equal")
ax2.set_xlim(-1.7, 0.8)
ax2.set_ylim(-1.6, 1.2)
ax2.set_xticks([-1, 0])
ax2.set_yticks([-1, 0, 1])
ax2.set_xlabel("$x_1$")
ax2.set_ylabel("$x_2$")
ax1.set_yticks([0, 1, 2])
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> - **`dendrogram(..., labels=np.arange(1, 10), leaf_font_size=10)`** — escreve, embaixo de cada folha, o número dado em `labels`, no tamanho `leaf_font_size`.
> - **`ax1.get_xticklabels()`** — os textos embaixo das folhas; **`rot.get_text()`** lê cada um, e **`rot.set_color`** muda a cor dele.
> - **`plt.subplots(..., height_ratios=[1, 1.3])`** — o gráfico de baixo fica 1,3 vez mais alto que o de cima.

### O algoritmo

As fusões seguem um algoritmo curto:

1. Comece com as $n$ observações, cada uma no seu próprio grupo, e com a dissimilaridade de cada par, as $n(n - 1)/2$ distâncias euclidianas, por exemplo.
2. Enquanto houver mais de um grupo:
   a. Entre todos os pares de grupos, ache os dois menos dissimilares e funda-os num só. A dissimilaridade entre eles é a altura da fusão no dendrograma.
   b. Calcule a dissimilaridade entre o grupo novo e cada um dos outros.

Quais pontos se juntam nas primeiras fusões dos nove pontos, e a que altura? As quatro primeiras linhas de `Z9` respondem, e a quinta mostra o que acontece depois:

In [ ]:
print("fusão: pontos · altura")
for f in range(4):
    a, b, h, _ = Z9[f]
    i, j = int(a) + 1, int(b) + 1
    print(f"{f + 1}.ª:", i, "e",
        j, "·", numero_br(h))
a, b = Z9[4, :2]
print("5.ª, na matriz:", int(a),
    "e", int(b))

Na matriz, as observações são numeradas de 0 a 8, e o `+ 1` as põe de 1 a 9. A partir de 9, um número indica um grupo formado antes: 9 é o grupo da 1.ª fusão, 10 o da 2.ª, e assim por diante.

As primeiras fusões juntam pares de pontos próximos, 1 e 6 primeiro e 5 e 7 logo depois. A quinta junta os grupos 9 e 10, isto é, $\{1, 6\}$ e $\{5, 7\}$, dois grupos de dois pontos.

O passo 2(b) esconde uma decisão: a dissimilaridade entre duas observações é a distância entre elas, mas qual é a dissimilaridade entre o grupo $\{1, 6\}$ e o grupo $\{5, 7\}$?

### A ligação

A **ligação** (*linkage*) define a dissimilaridade entre dois grupos $A$ e $B$ a partir das distâncias dos pares com uma observação de $A$ e uma de $B$. As quatro mais usadas:

| Ligação | Dissimilaridade |
|---|---|
| completa | maior distância entre pares |
| simples | menor distância entre pares |
| média | distância média entre pares |
| centroide | distância entre o centroide de $A$ e o de $B$ |

Com $A = \{1, 6\}$ e $B = \{5, 7\}$, há quatro pares com um ponto de cada grupo. Quanto vale a dissimilaridade entre os dois grupos em cada ligação?

In [ ]:
from sklearn.metrics import (
    pairwise_distances)

A = pontos[[0, 5]]
B = pontos[[4, 6]]
d = pairwise_distances(A, B)
print("completa:",
    numero_br(d.max()))
print("simples: ",
    numero_br(d.min()))
print("média:   ",
    numero_br(d.mean()))
cA = A.mean(axis=0)
cB = B.mean(axis=0)
dc = np.sqrt(((cA - cB)**2).sum())
print("centroide:",
    numero_br(dc))
igual = np.isclose(d.max(),
    Z9[4, 2])
print("completa = 5.ª fusão:",
    bool(igual))

> **🔧 Função**
>
> - **`pairwise_distances(A, B)`** — com dois argumentos, as distâncias entre cada linha de `A` e cada linha de `B`: aqui, uma tabela 2 × 2, os quatro pares com um ponto de cada grupo.
> - **`pontos[[0, 5]]`** — as linhas 0 e 5 do array, os pontos 1 e 6.
> - **`np.sqrt(v)`** — a raiz quadrada de cada elemento de `v`; aqui, da soma dos quadrados das diferenças entre as coordenadas dos dois centroides, a distância euclidiana entre eles.

Os dois grupos estão a 0,88 pela ligação completa, que olha o par mais afastado, e a 0,47 pela simples, que olha o mais próximo; a média e a centroide ficam no meio. A ligação completa dá a altura da quinta fusão da árvore, porque foi com ela que a árvore foi construída.

A ligação explica o corte em 8,5 dos 45 pontos, que deixou a classe 2 sozinha. Entre cada par de classes, a que distância ficam os pontos mais afastados, e os centroides?

In [ ]:
pares = [(0, 1), (1, 2), (0, 2)]
print("classes: completa",
    "· centroide")
for a, b in pares:
    Xa = X[classe == a]
    Xb = X[classe == b]
    dab = pairwise_distances(
        Xa, Xb)
    ca = Xa.mean(axis=0)
    cb = Xb.mean(axis=0)
    dc = np.sqrt(
        ((ca - cb)**2).sum())
    print(f"{a} e {b}:",
        numero_br(dab.max()), "·",
        numero_br(dc))

Pelos centroides, as classes 1 e 2 estão mais perto uma da outra, a 4,15, do que as classes 0 e 1, a 4,73. Pelo par mais afastado, que é o que a ligação completa olha, a ordem se inverte: 7,89 entre 0 e 1 e 8,15 entre 1 e 2. Por isso a árvore junta primeiro as classes 0 e 1, na fusão de altura 7,89, e o corte em 8,5 deixa a classe 2 sozinha. O 9,22 entre as classes 0 e 2 é a altura da raiz.

Com outra ligação, as distâncias entre grupos mudam, e as fusões e a árvore também. Nos 45 pontos, com o corte em três grupos, as quatro ligações dão os mesmos tamanhos?

In [ ]:
ligacoes = ["complete",
    "average", "single",
    "centroid"]
print("ligação: tamanhos")
for m in ligacoes:
    Zm = linkage(X, m)
    g = fcluster(Zm, 3,
        criterion="maxclust")
    cont = np.bincount(g)[1:]
    tam = np.sort(cont)
    print(f"  {m}:", *tam)
Zc = linkage(X, "centroid")
sobe = np.all(
    np.diff(Zc[:, 2]) >= 0)
print("centroide, alturas só",
    "sobem:", bool(sobe))

> **🔧 Função**
>
> - **`linkage(X, m)`** — o segundo argumento escolhe a ligação: `"complete"` (completa), `"average"` (média), `"single"` (simples) ou `"centroid"` (centroide).
> - **`np.bincount(g)[1:]`** — o `[1:]` descarta a contagem do 0, que o `fcluster` não usa como nome de grupo.

A completa, a média e a centroide dão três grupos de 15. A simples dá um grupo de 41, um de 3 e um de 1. Ela junta os grupos pelo par mais próximo, e, quanto mais pontos um grupo tem, mais chance de um deles estar perto de alguém de fora: o grupo grande cresce **encadeado** (*chaining*), e o corte deixa de fora só uns poucos pontos.

A centroide tem outro defeito. Numa fusão, o centroide do grupo novo pode ficar mais perto de um terceiro grupo do que os dois grupos estavam um do outro, e a fusão seguinte sai **mais baixa** que a anterior: é uma **inversão**, e o `False` impresso acima mostra que ela acontece nos 45 pontos. Num dendrograma, uma fusão assim fica abaixo de uma das fusões que ela junta, e a leitura pela altura deixa de valer. Por isso a centroide exige cuidado na leitura da árvore, embora seja comum em genômica. Entre as outras, preferem-se a completa e a média, que evitam o encadeamento extremo da simples e tendem a dar grupos de tamanhos mais parecidos.

### As amostras do `NCI60`

A tabela `NCI60` tem a expressão de milhares de genes em amostras de células de câncer. Com os genes padronizados, as ligações completa, média e simples, cortadas em 4 grupos, dividem as amostras do mesmo jeito?

In [ ]:
from sklearn.preprocessing \
    import StandardScaler

nci = pd.read_csv(
    "dados/NCI60.csv.gz")
tipo = nci["tipo"]
genes = nci.drop(columns="tipo")
print("forma:", genes.shape)
padronizador = StandardScaler()
padr = padronizador.fit_transform(
    genes)
arvores = {}
print("ligação: tamanhos")
for m in ligacoes[:3]:
    arvores[m] = linkage(padr, m)
    g = fcluster(arvores[m], 4,
        criterion="maxclust")
    cont = np.bincount(g)[1:]
    tam = np.sort(cont)
    print(f"  {m}:", *tam)

São 64 amostras e 6.830 genes. As três árvores dividem as mesmas amostras de jeitos bem diferentes. O maior grupo tem 40 amostras na completa, 54 na média e 61 na simples.

Na simples, os outros três grupos têm uma amostra cada, e a média também deixa duas amostras sozinhas, cada uma num grupo de 1. A média fica entre as duas, mais equilibrada que a simples, menos que a completa.

O encadeamento se vê nas fusões. Quando uma amostra sozinha se junta ao maior grupo, ele cresce uma amostra. Em quantas fusões o maior grupo cresce, e em quantas delas o que entra é uma amostra sozinha?

In [ ]:
n = len(padr)
print("maior grupo cresce · com")
print("uma amostra sozinha:")
for m in ligacoes[:3]:
    Zm = arvores[m]
    # tamanho do maior grupo
    # depois de cada fusão
    maior = np.maximum.accumulate(
        Zm[:, 3])
    cresce = np.diff(maior,
        prepend=1) > 0
    # números < n: amostra sozinha
    lados = Zm[:, :2] < n
    sos = lados.sum(axis=1)
    uma = cresce & (sos == 1)
    print(f"  {m}:", cresce.sum(),
        "·", uma.sum())

> **🔧 Função**
>
> - **`np.maximum.accumulate(v)`** — em cada posição, o maior valor de `v` até ali. **`np.diff(maior, prepend=1)`** põe um 1, o tamanho de todo grupo antes da primeira fusão, antes de tirar as diferenças.

Na simples, o maior grupo cresce em 39 fusões, 30 delas por uma amostra sozinha: o grupo grande se forma, na maior parte das vezes, uma amostra por vez. Na média, uma amostra sozinha faz o maior grupo crescer em 8 fusões, e na completa, em 2. Nos dendrogramas, com o corte em 4 grupos:

In [ ]:
# Figura: Os dendrogramas das 64 amostras do `NCI60`, com os genes padronizados e a distância euclidiana, pelas ligações completa, média e simples. A linha tracejada corta cada árvore em 4 grupos; os galhos abaixo dela têm a cor do seu grupo, e os de cima, em cinza esmaecido, ficam atrás da linha. As alturas não se comparam de um gráfico para outro: cada ligação mede a distância entre grupos de um jeito.
nomes = {"complete": "completa",
    "average": "média",
    "single": "simples"}
# cinza com 55% de opacidade
claro = "#7373738C"
apagado = dict(opcoes,
    above_threshold_color=claro)
fig, eixos = plt.subplots(
    3, 1, figsize=(3.8, 6.6))
for ax, m in zip(eixos,
        ligacoes[:3]):
    Zm = arvores[m]
    corte = (Zm[-4, 2]
        + Zm[-3, 2]) / 2
    dendrogram(Zm, ax=ax,
        color_threshold=corte,
        **apagado)
    for traco in ax.collections:
        traco.set_linewidth(0.8)
    ax.axhline(corte, ls="--",
        color="0.45", lw=1.2,
        zorder=3)
    ax.set_title(nomes[m],
        fontsize=10)
    ax.set_ylabel("altura")
    ax.grid(axis="x",
        visible=False)
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> - **`(Zm[-4, 2] + Zm[-3, 2]) / 2`** — a altura no meio entre a quarta e a terceira fusões mais altas. Depois das fusões até a quarta mais alta, sobram 4 grupos; o corte nessa altura os separa.
> - **`ax.collections`** — os conjuntos de traços do gráfico: o `dendrogram` desenha os galhos de cada cor como um conjunto. **`traco.set_linewidth(0.8)`** afina os traços, que com 64 folhas se amontoariam.
> - **`"#7373738C"`** — uma cor em hexadecimal: os seis primeiros dígitos são o cinza, e os dois últimos, a opacidade, 55%. **`dict(opcoes, ...)`** copia `opcoes` com a cor dos galhos acima do corte trocada por essa.

Os grupos da ligação completa têm relação com o tipo de câncer? Na coluna `tipo`, que não entrou no agrupamento, os rótulos terminados em `-repro` são medições repetidas de duas linhagens, a K562, de leucemia, e a MCF7, de mama, e `desconhecido` é uma amostra de origem não identificada. As seis fusões mais altas, e os saltos entre cada uma e a de baixo, mostram onde a árvore sugere cortar; o corte em 140 dá 4 grupos, e a tabela cruzada os compara com `tipo`.

In [ ]:
completa = arvores["complete"]
print("fusões mais altas:")
for k in range(1, 7):
    h = completa[-k, 2]
    print(f"  {k}.ª:",
        numero_br(h, 1))
# salto k: altura da k.ª menos
# a da (k + 1).ª
# alturas como impressas
altas = completa[-6:, 2].round(1)
saltos = np.diff(altas)[::-1]
print("saltos, da 1.ª à 5.ª:")
print(" ", *[numero_br(s, 1)
    for s in saltos])
em_ordem = np.sort(saltos)[::-1]
print("em ordem:",
    *[numero_br(s, 1)
    for s in em_ordem])
g4 = fcluster(completa, 140,
    criterion="distance")
print("grupos no corte 140:",
    len(np.unique(g4)))
g_max = fcluster(completa, 4,
    criterion="maxclust")
mesma = adjusted_rand_score(
    g4, g_max) == 1
print("os do corte em 4 grupos:",
    bool(mesma))
tab = pd.crosstab(tipo, g4,
    colnames=["grupo"])
print(tab)

O 140 cai entre a 4.ª e a 3.ª fusões mais altas, a 138,7 e a 142,4, e dá os mesmos 4 grupos do corte em 4. O maior salto, 19,4, três vezes o seguinte, é o da última fusão, de 144,1 para 163,5, e um corte nele dá 2 grupos. O seguinte, 6,4, entre a 5.ª e a 4.ª, daria 5. O 4 não cai num salto: entre a 4.ª e a 3.ª, a diferença é de 3,7. Ele é uma escolha de quem analisa, feita aqui para comparar os grupos com `tipo`.

A tabela tem uma linha por tipo, e os números das colunas são só nomes. Onde caem a leucemia, o melanoma e a mama?

In [ ]:
leu = tab.loc["leucemia"]
g_leu = leu.idxmax()
col = tab[g_leu]
print("leucemia no mesmo grupo:",
    leu.max(), "de", leu.sum())
print("o grupo tem:",
    col.sum(), "amostras")
outros = col[col > 0].index
print(" ", *outros)
mel = tab.loc["melanoma"]
g_mel = mel.idxmax()
col = tab[g_mel]
print("melanoma no mesmo grupo:",
    mel.max(), "de", mel.sum())
print("  tamanho do grupo:",
    col.sum())
print("  tipos no grupo:",
    (col > 0).sum())
mama = tab.loc["mama"]
print("grupos com mama:",
    (mama > 0).sum())

> **🔧 Função**
>
> - **`tab[g_leu]`** — a coluna desse grupo, com quantas amostras de cada tipo ele tem; **`col[col > 0].index`**, os tipos que aparecem nele.

As 6 amostras de leucemia ficam num grupo de 8, com apenas mais duas: as medições repetidas da K562, também de leucemia.

As 8 de melanoma ficam todas no grupo de 40, mas esse grupo junta 9 tipos da coluna `tipo`. As de mama se espalham por 3 grupos. O agrupamento, que não viu a coluna `tipo`, junta a leucemia num grupo e mistura os tipos nos outros.

### A medida de dissimilaridade

Até aqui, a dissimilaridade entre duas observações foi a distância euclidiana. Uma loja virtual agrupa seus clientes pelo número de compras de cada produto. Dois clientes compram os mesmos produtos, um em grande volume e o outro pouco. Eles são parecidos? Pela distância euclidiana, os valores de um ficam longe dos do outro, e ela tende a juntar os clientes que compram pouco, seja o que for que comprem.

Três perfis simulados, com $p = 20$ variáveis, põem a mesma pergunta. O perfil 2 tem a mesma forma do perfil 1, em outro nível e com o dobro da amplitude; o perfil 3 fica no nível do 1, com outra forma. Qual dos dois é mais parecido com o perfil 1?

In [ ]:
ger = np.random.default_rng(15)
base = ger.normal(size=20)
ruido = ger.normal(size=20)
obs1 = 5 + base
obs2 = 12 + 2 * base + 0.3 * ruido
obs3 = 5 + ger.normal(size=20)
P = np.vstack([obs1, obs2, obs3])
print("forma:", P.shape)

> **🔧 Função**
>
> - **`np.vstack([obs1, obs2, obs3])`** — empilha os três perfis como as linhas de uma tabela 3 × 20.

In [ ]:
# Figura: Os três perfis simulados, com o valor de cada uma das 20 variáveis. Os perfis 1 e 3 estão no mesmo nível, e os perfis 1 e 2 sobem e descem juntos.
var = np.arange(1, 21)
fig, ax = plt.subplots(
    figsize=(3.8, 2.8))
for k, cor in enumerate(
        ["C0", "C1", "C2"]):
    ax.plot(var, P[k], "o-",
        color=cor, markersize=3,
        linewidth=1,
        label=f"perfil {k + 1}")
ax.set_xlabel("variável")
ax.set_ylabel("valor")
ax.set_xticks([1, 5, 10, 15, 20])
ax.set_yticks([5, 10, 15])
ax.set_xlim(0, 21)
ax.legend(ncol=3, fontsize=9,
    loc="lower center",
    bbox_to_anchor=(0.5, 1),
    frameon=False)
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> - **`ax.plot(..., label=...)`** e **`ax.legend(...)`** — cada linha recebe um nome, e a legenda os mostra; `ncol=3` põe os três lado a lado, e `bbox_to_anchor=(0.5, 1)` com `loc="lower center"` a coloca acima do gráfico, fora da área dos dados.

A resposta depende do que se quer que "parecido" signifique. A **distância baseada em correlação** trata como parecidas duas observações cujos valores sobem e descem juntos ao longo das variáveis, mesmo que estejam em níveis diferentes. É um uso incomum da correlação: em vez de correlacionar duas variáveis ao longo das observações, correlaciona duas observações, o **perfil** de cada uma, ao longo das $p$ variáveis. A distância é $1 - r$, em que $r$ é essa correlação. Quanto vale cada medida entre os três perfis?

In [ ]:
d_eu = pairwise_distances(P)
d_co = pairwise_distances(P,
    metric="correlation")
print("par: euclidiana · 1 − r")
for a, b in [(0, 1), (0, 2),
        (1, 2)]:
    e = numero_br(d_eu[a, b])
    c = numero_br(d_co[a, b], 3)
    print(f"{a + 1} e {b + 1}:",
        e, "·", c)

> **🔧 Função**
>
> - **`pairwise_distances(P, metric="correlation")`** — com `metric="correlation"`, $1 - r$ entre cada par de linhas no lugar da distância euclidiana, com $r$ a correlação entre os 20 valores de uma linha e os 20 da outra.

Pela distância euclidiana, o perfil 1 fica perto do 3, a 5,83, e longe do 2, a 32,48. Pela correlação, é o contrário: $1 - r$ vale 0,012 entre 1 e 2 e 0,863 entre 1 e 3. Na loja, com a correlação, os clientes que compram os mesmos produtos ficam juntos, em volume grande ou pequeno.

As duas medidas se ligam. Se o perfil de cada observação for padronizado, com média 0 e desvio padrão 1 nas suas $p$ variáveis, as duas dão a mesma ordem entre os pares: $1 - r$ é igual à distância euclidiana ao quadrado dividida por $2p$, com o desvio padrão calculado com divisor $p$. Vale nos estados do `USArrests`? As quatro variáveis de lá têm unidades diferentes, e padronizar cada estado ao longo delas não é uma boa prática para agrupar: aqui, a padronização por linha serve só para conferir a igualdade.

In [ ]:
prisoes = pd.read_csv(
    "dados/USArrests.csv",
    index_col="estado")
T = prisoes.to_numpy()
print("forma:", T.shape)
media = T.mean(axis=1,
    keepdims=True)
dp = T.std(axis=1,
    keepdims=True)
Tp = (T - media) / dp
d2 = pairwise_distances(Tp)**2
um_r = pairwise_distances(Tp,
    metric="correlation")
p = T.shape[1]
print("1 − r = d² / (2p):",
    bool(np.allclose(um_r,
        d2 / (2 * p))))

> **🔧 Função**
>
> - **`T.mean(axis=1, keepdims=True)`** — a média de cada **linha**, e não de cada coluna; `keepdims=True` a mantém como coluna, 50 × 1, para a subtração valer linha a linha. **`T.std(axis=1, ...)`**, do mesmo jeito, o desvio padrão de cada linha, com divisor $p$.

São 50 estados e $p = 4$ variáveis. A distância baseada em correlação é o quadrado da distância euclidiana dividido por $2p$, depois de tirar de cada perfil o seu nível e a sua amplitude. É o que ela ignora nos perfis 1 e 2.

No `NCI60`, a ligação completa com a distância baseada em correlação, cortada em 4 grupos, divide as amostras como a distância euclidiana?

In [ ]:
Zr = linkage(padr, "complete",
    metric="correlation")
g4r = fcluster(Zr, 4,
    criterion="maxclust")
cont = np.bincount(g4r)[1:]
print("tamanhos:", *np.sort(cont))
ari = adjusted_rand_score(g4,
    g4r)
print("comparada à euclidiana:",
    numero_br(ari, 3))
leu = g4r[tipo == "leucemia"]
print("leucemia num grupo só:",
    len(np.unique(leu)) == 1)

> **🔧 Função**
>
> - **`linkage(padr, "complete", metric="correlation")`** — a mesma ligação completa, com $1 - r$ entre as linhas no lugar da distância euclidiana.

Com a correlação, os tamanhos dos grupos mudam, e o índice de Rand ajustado entre as duas divisões dá 0,186, bem mais perto de 0 que de 1: as duas medidas dividem as amostras de jeitos diferentes. As 6 de leucemia continuam juntas num grupo.

Uma única árvore dá as divisões aninhadas em qualquer número de grupos, de 1 a $n$, e o número se escolhe depois, pela altura do corte; um salto grande nas alturas é indicação, não regra. A árvore depende da ligação e da medida de dissimilaridade, escolhidas antes, e o corte, como o $K$ do *k*-means, fica com quem analisa.

## Questões Práticas em Clustering

> **📌 Nota**
>
> Esta seção corresponde às seções 12.4.3 e 12.5.4 de James et al. (2023).

Para agrupar observações, alguém decide se padroniza as colunas, quantos grupos pede, que ligação usa e com que semente começa. Cada decisão dessas é pequena e pode mudar os grupos. Se outra tivesse sido tomada, os grupos continuariam lá?

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("estilo-figuras.mplstyle")

A função `numero_br` escreve os números com a pontuação brasileira:

In [ ]:
numero_br = lambda v, casas=2: (
    f"{v:,.{casas}f}"
    .replace(",", "_")
    .replace(".", ",")
    .replace("_", ".")
    .replace("-", "−")
)

### Padronizar ou não

Os 50 estados do `USArrests` têm quatro colunas: as prisões por homicídio, agressão e estupro a cada 100.000 habitantes, e o percentual de população urbana. Com a ligação completa e três grupos, o agrupamento hierárquico dá a mesma divisão com as colunas como vêm e com elas padronizadas?

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import (
    AgglomerativeClustering)
from sklearn.preprocessing \
    import StandardScaler

prisoes = pd.read_csv(
    "dados/USArrests.csv",
    index_col="estado")
padronizador = StandardScaler()
padr = padronizador.fit_transform(
    prisoes)
print("forma:", padr.shape)
ag = AgglomerativeClustering(
    n_clusters=3,
    linkage="complete")
g_cru = ag.fit_predict(prisoes)
g_pad = ag.fit_predict(padr)
print("tamanhos dos 3 grupos:")
print("  sem padronizar:",
    *np.sort(np.bincount(g_cru)))
print("  padronizado:",
    *np.sort(np.bincount(g_pad)))

> **🔧 Função**
>
> - **`ag.fit_predict(X)`** — ajusta o agrupamento às linhas de `X` e devolve o grupo de cada uma, o mesmo que `ag.fit(X)` seguido de `ag.labels_`. O mesmo `ag` serve para as duas tabelas: cada chamada refaz o agrupamento do zero.

Os tamanhos já mostram que as divisões diferem. O índice de Rand ajustado, que a saída abrevia como ARI, mede quanto, e as faixas de `agressao` em cada grupo mostram o que a divisão sem padronizar separou:

In [ ]:
from sklearn.metrics import (
    adjusted_rand_score)

ari = adjusted_rand_score(
    g_cru, g_pad)
print("ARI entre as duas:",
    numero_br(ari, 3))
agr = prisoes["agressao"]
for nome, g in [("sem", g_cru),
        ("com", g_pad)]:
    fx = (agr.groupby(g)
        .agg(["min", "max"])
        .sort_values("min"))
    lo = fx["min"].to_numpy()
    hi = fx["max"].to_numpy()
    sep = np.all(hi[:-1] < lo[1:])
    print(f"{nome} padronizar:")
    for a, b in zip(lo, hi):
        print(f"  {a} a {b}")
    print("  faixas separadas:",
        bool(sep))

> **🔧 Função**
>
> - **`serie.groupby(g).agg(["min", "max"])`** — separa os valores da série pelo grupo `g` e calcula, em cada grupo, o menor e o maior: uma linha por grupo, com as colunas `min` e `max`. **`.sort_values("min")`** ordena as linhas pela coluna `min`.
> - **`hi[:-1] < lo[1:]`** — compara o máximo de cada faixa com o mínimo da faixa seguinte; tudo `True` quer dizer que nenhuma faixa invade a seguinte.

O índice de Rand ajustado entre as duas divisões é 0,368, longe de 1. Sem padronizar, os três grupos são só três faixas de `agressao`, que não se tocam. Padronizadas as colunas, as faixas se sobrepõem, e as outras três colunas passam a pesar. A figura abaixo mostra os mesmos 50 estados duas vezes, com a cor do grupo de cada divisão:

In [ ]:
# Figura: Os 50 estados do `USArrests`, com as prisões por agressão no eixo horizontal e o percentual de população urbana no vertical, nos dois painéis. A cor e a marca de cada ponto são as do seu grupo na ligação completa com 3 grupos: em cima, com as colunas como vêm; embaixo, com as colunas padronizadas. As cores são só nomes para os grupos.
import matplotlib.pyplot as plt

fig, eixos = plt.subplots(
    2, 1, figsize=(3.8, 5.2),
    sharex=True)
urb = prisoes["pop_urbana"]
cores = ["C0", "C1", "C2"]
marcas = ["o", "s", "^"]
titulos = ["sem padronizar",
    "padronizado"]
for ax, g, tit in zip(eixos,
        [g_cru, g_pad], titulos):
    ordem = (agr.groupby(g).mean()
        .sort_values().index)
    for k, grupo in enumerate(
            ordem):
        m = g == grupo
        ax.scatter(agr[m], urb[m],
            s=22,
            marker=marcas[k],
            facecolors="none",
            edgecolors=cores[k],
            linewidths=1.1)
    ax.set_title(tit)
    ax.set_ylabel(
        "pop. urbana (%)")
eixos[1].set_xlabel(
    "agressão (por 100 mil)")
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> - **`ax.scatter(..., facecolors="none", edgecolors=cor)`** — pontos vazados, só com o contorno na cor dada; pontos que caem no mesmo lugar continuam visíveis.

Por que uma coluna manda? Com $n$ observações e $p$ colunas, a distância euclidiana ao quadrado entre as observações $i$ e $i'$ é uma soma com uma parcela por coluna:

$$
d^2(i, i') = \sum_{j=1}^{p} (x_{ij} - x_{i'j})^2.
$$

Somada em todos os pares ordenados $(i, i')$ de estados, incluindo $i = i'$, a parcela da coluna $j$ dá $2n^2$ vezes a variância dela, calculada com o divisor $n$. A fatia de cada coluna na soma das distâncias ao quadrado é, então, a fração dela na variância total, a mesma conta que a seção 15.3 fez para `agressao`:

In [ ]:
def fatias(A):
    A = np.asarray(A, dtype=float)
    v = A.var(axis=0)
    return v / v.sum()

amplitude = (prisoes.max()
    - prisoes.min())
f_cru = fatias(prisoes)
f_pad = fatias(padr)
print("amplitude · fatia crua",
    "· padr.")
for c, amp, a, b in zip(
        prisoes.columns,
        amplitude, f_cru, f_pad):
    print(f"  {c}:",
        numero_br(amp, 1), "·",
        numero_br(a), "·",
        numero_br(b))
outras = amplitude.drop(
    "agressao")
vezes = np.all(
    amplitude["agressao"]
    > 4 * outras)
print("agressao > 4 × as outras:",
    bool(vezes))

> **🔧 Função**
>
> - **`np.asarray(A, dtype=float)`** — `A` como array de números decimais, seja ela uma tabela do `pandas` ou já um array.
> - **`prisoes.max() - prisoes.min()`** — a amplitude de cada coluna: o maior valor menos o menor.

Sem padronizar, `agressao` responde por 0,96 da soma das distâncias ao quadrado. A amplitude dela, 292,0, passa de quatro vezes a de cada uma das outras três (homicídio, 16,6; população urbana, 59,0; estupro, 38,7), e a parcela de cada coluna cresce com o quadrado das diferenças: as diferenças em `agressao` abafam as das outras colunas. Padronizadas, as quatro colunas ficam com 0,25 cada, e pesam igual.

Padronizar é sempre o certo? Uma loja vende meias e computadores e quer agrupar os clientes pelo que compram. Oito clientes compraram, num ano, estas quantidades:

In [ ]:
meias = np.array(
    [8, 11, 7, 6, 5, 6, 7, 8])
pcs = np.array(
    [0, 0, 0, 0, 1, 1, 1, 1])
qtd = np.column_stack(
    [meias, pcs])
# preço: 20 reais o par de meias
# e 4.000 reais o computador
reais = qtd * [20, 4000]
pad = StandardScaler() \
    .fit_transform(qtd)
versoes = {"quantidade": qtd,
    "padronizada": pad,
    "em reais": reais}
print("fatia das meias:")
for nome, A in versoes.items():
    f = fatias(A)[0]
    print(f"  {nome}:",
        numero_br(f, 3))

> **🔧 Função**
>
> - **`np.column_stack([a, b])`** — junta os arrays `a` e `b` como as colunas de uma tabela, aqui de 8 linhas e 2 colunas.
> - **`qtd * [20, 4000]`** — multiplica a primeira coluna por 20 e a segunda por 4.000, linha a linha.

Contadas em unidades, as meias ficam com 0,922 da soma das distâncias ao quadrado: um cliente que comprou 11 pares fica longe de um que comprou 5, e ter comprado ou não um computador quase não conta. Padronizadas, as duas colunas pesam o mesmo, 0,500. Em reais, a fatia das meias fica abaixo de 0,001, e quem decide as distâncias é o computador, que vale muito mais para a loja.

Qual das três versões está certa depende do que a loja quer dos grupos. Se o que importa é o dinheiro gasto, a versão em reais serve; se é o hábito de compra, a padronizada dá a meias e computadores o mesmo peso. Quando as colunas estão em unidades que não se comparam, como prisões por 100.000 habitantes e percentual de população urbana, padronizar evita que a unidade escolha os grupos. O *k*-means, que usa essa distância, tem a mesma questão.

### Quantos grupos

O *k*-means pede $K$ antes de começar. A inércia, a soma das distâncias ao quadrado de cada ponto ao centroide do seu grupo, serve para escolher $K$? Para comparar, a conta roda nos estados padronizados e em duas tabelas de 50 linhas e 4 colunas, com valores sorteados de uma distribuição normal e sem grupo nenhum. Na primeira, o ruído independente, as colunas não têm relação entre si. Na segunda, o ruído correlacionado, a normal tem a matriz de covariância dos estados padronizados, estimada por `np.cov`, e as colunas se correlacionam como as dos estados.

In [ ]:
from sklearn.cluster import KMeans

ger_ind = (np.random
    .default_rng(15))
ruido = ger_ind.normal(
    size=(50, 4))
cov = np.cov(padr.T)
ger_cor = (np.random
    .default_rng(15))
nuvem = (ger_cor
    .multivariate_normal(
        np.zeros(4), cov, 50))
tabelas = {"estados": padr,
    "independente": ruido,
    "correlacionado": nuvem}
Ks = range(1, 9)
iner = {}
for nome, A in tabelas.items():
    iner[nome] = np.array([
        KMeans(n_clusters=K,
            n_init=50,
            random_state=15)
        .fit(A).inertia_
        for K in Ks])
print("inércia com K = 1:")
for nome, v in iner.items():
    print(f"  {nome}:",
        numero_br(v[0]))
    cai = np.all(np.diff(v) < 0)
    print("    só cai:",
        bool(cai))
rel = {nome: v / v[0]
    for nome, v in iner.items()}
print("fração que resta:")
print("K: estados · indep.",
    "· correl.")
for i, K in enumerate(Ks):
    print(f"{K}:", *[numero_br(
        rel[n][i]) for n in rel])
ind = rel["independente"][1:]
teto = np.maximum(
    rel["estados"][1:],
    rel["correlacionado"][1:])
acima = np.all(ind > teto)
print("independente acima das")
print("  outras, em K > 1:",
    bool(acima))

> **🔧 Função**
>
> - **`np.cov(padr.T)`** — a matriz de covariância das colunas de `padr`: na diagonal, a variância de cada coluna; fora dela, a covariância de cada par. O `.T` transpõe a tabela, porque `np.cov` trata cada linha como uma variável. O divisor é $n - 1$, e não o $n$ do `StandardScaler`.
> - **`ger.multivariate_normal(media, cov, 50)`** — sorteia 50 linhas de uma normal com o vetor de médias e a matriz de covariância dados. Aqui o vetor de médias é **`np.zeros(4)`**, quatro médias iguais a 0.
> - **`np.maximum(a, b)`** — o maior de `a` e `b` em cada posição: aqui, a curva mais alta entre a dos estados e a do ruído correlacionado, em cada $K$.

Com $K = 1$, o centroide é a média de cada coluna. Nos estados padronizados, cada coluna tem média 0 e, com o divisor $n$ do `StandardScaler`, soma dos quadrados igual a $n$: a inércia é $np$, com $n = 50$ estados e $p = 4$ colunas, 200. Os dois ruídos não foram padronizados, e as inércias deles com $K = 1$ são 197,62 e 216,12.

A inércia cai a cada grupo a mais, nas três tabelas. Como na seção 15.4, a melhor divisão com $K + 1$ grupos nunca tem inércia maior que a melhor com $K$, porque dá para partir um grupo em dois sem aumentar a soma; aqui, os valores que o *k*-means achou também caem. O menor valor fica no maior $K$ testado, e por isso a inércia não escolhe $K$ pelo menor valor.

Uma leitura comum é procurar um cotovelo, um $K$ a partir do qual a inércia passa a cair pouco. Na figura abaixo, as três curvas estão divididas pela inércia de $K = 1$, para ficarem na mesma escala:

In [ ]:
# Figura: A inércia do *k*-means contra o número de grupos $K$, dividida pela inércia de $K = 1$, nos 50 estados padronizados do `USArrests` e em duas tabelas de ruído normal de 50 linhas e 4 colunas, sem grupo nenhum: uma com colunas independentes e outra com a matriz de covariância dos estados.
fig, ax = plt.subplots(
    figsize=(3.8, 3.0))
estilos = {
    "estados": ("C0", "o"),
    "correlacionado": ("C1", "^"),
    "independente": ("0.45", "s")}
for nome, (cor, mk) in \
        estilos.items():
    ax.plot(Ks, rel[nome],
        marker=mk, color=cor,
        markersize=4, label=nome)
ax.set_xlim(0.6, 8.4)
ax.set_ylim(0, 1.05)
ax.set_xticks(list(Ks))
ax.yaxis.set_major_formatter(
    lambda v, _: f"{v:.1f}"
    .replace(".", ","))
ax.set_xlabel("K")
ax.set_ylabel("fração da inércia")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

Com $K = 4$, restam 0,29 da inércia nos estados e 0,50 no ruído independente: a curva dos estados cai mais depressa. Isso não prova que há grupos nos estados.

O ruído correlacionado, que também não tem grupo nenhum, deixa 0,35 com $K = 4$. Nestes dois sorteios, a correlação entre as colunas responde por 0,15, de 0,50 para 0,35, dos 0,21 que separam o ruído independente, com 0,50, dos estados, com 0,29. A queda rápida vem, em boa parte, de as colunas andarem juntas, e não de grupos. Os 0,06 que ainda separam o ruído correlacionado, com 0,35, dos estados, com 0,29, vêm de um sorteio só e não permitem afirmar que os estados têm grupos.

Onde fica o cotovelo dos estados, em 2 ou em 4? A curva não responde. E o ruído independente, sem grupo nenhum, também aceita ser dividido:

In [ ]:
km3 = KMeans(n_clusters=3,
    n_init=50, random_state=15)
km3.fit(ruido)
tam = np.sort(
    np.bincount(km3.labels_))
print("ruído, K = 3, tamanhos:",
    *tam)
print("pontos sem grupo:",
    len(ruido) - tam.sum())

Com $K = 3$, o *k*-means divide o ruído em grupos de 15, 16 e 19 pontos, e todo ponto entra num deles. Pedidos três grupos, ele devolve três, haja ou não grupos no dado. A escolha de $K$, como o corte do dendrograma, é de quem analisa: vale olhar alguns valores e ficar com o que dá grupos que se interpretam e que se repetem.

### Rótulos e sementes

O *k*-means com $K = 4$ roda duas vezes nos estados padronizados, com as sementes 15 e 16. Quantos estados recebem o mesmo rótulo nas duas?

In [ ]:
km_a = KMeans(n_clusters=4,
    n_init=50, random_state=15)
km_b = KMeans(n_clusters=4,
    n_init=50, random_state=16)
rot_a = km_a.fit(padr).labels_
rot_b = km_b.fit(padr).labels_
iguais = np.mean(rot_a == rot_b)
print("rótulos iguais:",
    numero_br(iguais))
print(pd.crosstab(rot_a, rot_b,
    rownames=["semente 15"],
    colnames=["semente 16"]))
print("ARI:", numero_br(
    adjusted_rand_score(
        rot_a, rot_b)))
mesma = np.isclose(km_a.inertia_,
    km_b.inertia_)
print("mesma inércia:",
    bool(mesma))

> **🔧 Função**
>
> - **`np.mean(a == b)`** — a fração das posições em que `a` e `b` coincidem: cada `True` conta 1, e cada `False`, 0.

Só 0,48 dos estados têm o mesmo rótulo nas duas execuções, e a divisão, ainda assim, é a mesma: cada linha da tabela cruzada tem uma só casa preenchida, o índice de Rand ajustado é 1,00, e a inércia é igual. As duas sementes acharam os mesmos quatro grupos e os numeraram em outra ordem. Os rótulos são nomes, e comparar dois agrupamentos rótulo a rótulo mede a numeração; a tabela cruzada e o índice de Rand ajustado comparam os grupos.

No `NCI60`, as 64 amostras de câncer têm 6.830 genes. Com os genes padronizados, $K = 4$ e 100 começos por semente, dez sementes chegam à mesma divisão?

In [ ]:
nci = pd.read_csv(
    "dados/NCI60.csv.gz")
tipo = nci["tipo"]
genes = nci.drop(columns="tipo")
print("forma:", genes.shape)
padr_nci = (padronizador
    .fit_transform(genes))

def dez_sementes(A):
    return [KMeans(n_clusters=4,
        n_init=100,
        random_state=s).fit(A)
        for s in range(10)]

def distintas(modelos):
    unicas = []
    for m in modelos:
        r = m.labels_
        nova = all(
            adjusted_rand_score(
                r, u) < 1
            for u in unicas)
        if nova:
            unicas.append(r)
    return len(unicas)

km_genes = dez_sementes(padr_nci)
print("genes, 10 sementes:")
print("  divisões distintas:",
    distintas(km_genes))
iner_nci = [m.inertia_
    for m in km_genes]
print("  inércia, menor:",
    numero_br(min(iner_nci), 0))
print("  inércia, maior:",
    numero_br(max(iner_nci), 0))
r0 = km_genes[0].labels_
aris_0 = [adjusted_rand_score(
    r0, m.labels_)
    for m in km_genes[1:]]
print("  menor ARI com a 0:",
    numero_br(min(aris_0)))

A função `distintas` guarda uma divisão como nova quando o índice de Rand ajustado dela com cada uma das já guardadas fica abaixo de 1, isto é, quando ela não repete nenhuma.

As dez sementes dão 9 divisões diferentes, com inércias entre 350.576 e 351.928. As inércias são próximas, e as divisões, não.

Com a semente 0 como referência, o menor índice de Rand ajustado entre as outras nove é 0,50. Há vários ótimos locais de inércia parecida e divisão diferente, e 100 começos por semente não bastam para as dez execuções coincidirem. O que fazer com elas? Ficar com a de menor inércia é o que o `n_init` já faz dentro de cada semente; mais útil é conferir se o que se quer concluir depende da divisão escolhida, como a parte sobre hierárquico e *k*-means, adiante, faz com o grupo da leucemia.

Os primeiros escores da `PCA` resumem a tabela e deixam de fora a variação pequena, que às vezes é ruído (seção 15.3). O mesmo *k*-means, rodado sobre os 5 primeiros escores em vez dos 6.830 genes, chega à mesma divisão nas dez sementes?

In [ ]:
from sklearn.decomposition \
    import PCA

pca = PCA()
escores = pca.fit_transform(
    padr_nci)
cinco = escores[:, :5]
print("forma:", cinco.shape)
acum = (pca
    .explained_variance_ratio_[:5]
    .sum())
print("variância nos 5:",
    numero_br(acum))
km_esc = dez_sementes(cinco)
print("5 escores, 10 sementes:")
print("  divisões distintas:",
    distintas(km_esc))

> **🔧 Função**
>
> - **`escores[:, :5]`** — as 5 primeiras colunas dos escores, as das 5 primeiras componentes, para as 64 amostras.

Sobre os 5 escores, as dez sementes dão uma divisão só. Os 5 escores guardam 0,32 da variância total dos genes: são um resumo curto, e o número 5 é mais uma decisão pequena, como $K$, que outro valor poderia mudar. O problema ficou menor, com 5 colunas em vez de 6.830, e os ótimos locais que atrapalhavam deixaram de aparecer nessas dez sementes.

### Hierárquico, *k*-means e escores no `NCI60`

A ligação completa, cortada em 4 grupos, e o *k*-means com $K = 4$ dividem as amostras do mesmo jeito? Os números dos grupos são nomes arbitrários, e a função `por_tamanho` troca cada um pelo tamanho do grupo, que tem significado: "o grupo de 8" é o mesmo, qualquer que seja o número que o método lhe deu.

In [ ]:
def por_tamanho(rot):
    cont = np.bincount(rot)
    distintos = len(set(cont))
    assert distintos == len(cont)
    return cont[rot]

ag4 = AgglomerativeClustering(
    n_clusters=4,
    linkage="complete")
hc = por_tamanho(
    ag4.fit_predict(padr_nci))
km15 = KMeans(n_clusters=4,
    n_init=100, random_state=15)
km_tam = por_tamanho(
    km15.fit(padr_nci).labels_)
print(pd.crosstab(hc, km_tam,
    rownames=["hierárquico"],
    colnames=["k-means"]))
ari_hk = adjusted_rand_score(
    hc, km_tam)
print("ARI:",
    numero_br(ari_hk, 3))

> **🔧 Função**
>
> - **`cont[rot]`** — troca cada rótulo pelo tamanho do seu grupo, como `centros[nuvem]` trocou cada nuvem pelo seu centro na seção 15.4. **`assert distintos == len(cont)`** interrompe a execução com um erro se dois grupos tiverem o mesmo tamanho, porque aí o tamanho não distinguiria um do outro.

As linhas são os grupos do hierárquico, e as colunas, os do *k*-means, cada um com o seu tamanho. O índice de Rand ajustado, 0,314, diz que as duas divisões diferem bastante. O grupo de 7 do hierárquico está inteiro no grupo de 32 do *k*-means, e o grupo de 40 se espalha por três grupos do *k*-means. E as amostras de leucemia?

In [ ]:
e_leu = tipo == "leucemia"
leu = np.unique(hc[e_leu])
print("leucemia num grupo só:",
    len(leu) == 1)
g_leu = hc == leu[0]
print("tamanho do grupo:", leu[0])
print(tipo[g_leu].value_counts())

As amostras de leucemia ficam num grupo só do hierárquico, o de 8, que a seção 15.5 já descreveu: as de leucemia e as K562-repro, medições repetidas de uma linhagem também de leucemia. Outra divisão tem exatamente esse grupo quando as 8 amostras dele recebem o mesmo rótulo e nenhuma outra amostra recebe esse rótulo. É o que a função `tem_o_grupo` confere em três divisões: a do *k*-means com a semente 15, as das dez sementes de `km_genes` e a do hierárquico com a ligação média no lugar da completa, porque a ligação também é uma decisão:

In [ ]:
def tem_o_grupo(rot):
    r = rot[g_leu]
    so_um = len(set(r)) == 1
    n = (rot == r[0]).sum()
    return bool(so_um
        and n == g_leu.sum())

ag_med = AgglomerativeClustering(
    n_clusters=4,
    linkage="average")
hc_media = ag_med.fit_predict(
    padr_nci)
n_tem = sum(tem_o_grupo(
    m.labels_) for m in km_genes)
print("k-means, semente 15:",
    tem_o_grupo(km_tam))
print("10 sementes:", n_tem)
print("ligação média:",
    tem_o_grupo(hc_media))
print("  tamanhos:", *np.sort(
    np.bincount(hc_media)))
ari_lig = adjusted_rand_score(
    hc, hc_media)
print("  ARI com a completa:",
    numero_br(ari_lig, 3))

> **🔧 Função**
>
> - **`rot[g_leu]`** — os rótulos só das 8 amostras do grupo da leucemia, porque `g_leu` é `True` nelas e `False` nas outras.

O *k*-means com a semente 15 forma exatamente esse grupo, e 7 das dez sementes o formam. A ligação média também o forma, mas divide o resto em grupos de 1, 1 e 54 amostras, e por isso o índice de Rand ajustado com a completa fica em 0,362.

O hierárquico também pode rodar sobre os 5 primeiros escores, em vez dos genes. Que grupos ele forma, e como eles se comparam com os tipos de câncer, que não entraram no agrupamento?

In [ ]:
esc = por_tamanho(
    ag4.fit_predict(cinco))
tab = pd.crosstab(tipo, esc,
    colnames=["grupo"])
print(tab)
colon = (tab.loc["cólon"] > 0)
print("cólon num grupo só:",
    colon.sum() == 1)
leuc = (tab.loc["leucemia"] > 0)
print("grupos com leucemia:",
    leuc.sum())
print("ARI com o dos genes:",
    numero_br(adjusted_rand_score(
        hc, esc), 3))

Sobre os escores, os grupos têm 4, 6, 20 e 34 amostras, e a leucemia deixou de formar um grupo só: parte dela fica no grupo de 6, com as duas K562-repro, e parte no de 34.

Os grupos também não coincidem com os tipos. As 7 de cólon ficam juntas, no grupo de 34, mas a mama se divide entre os grupos de 4 e de 20, e o SNC, entre os de 20 e de 34.

O índice de Rand ajustado com a divisão feita sobre os genes é 0,270, menor que os 0,314 entre hierárquico e *k*-means nos genes: trocar a tabela de entrada mudou os grupos ao menos tanto quanto trocar o método.

A figura abaixo mostra as 64 amostras nas duas primeiras componentes, com a cor e a marca do grupo formado sobre os 5 escores. O agrupamento usou 5 dimensões, e o gráfico mostra 2: grupos que se separam nas outras três podem se sobrepor aqui.

In [ ]:
# Figura: Os escores das 64 amostras do `NCI60` nas duas primeiras componentes principais, com a cor e a marca do grupo do agrupamento hierárquico, pela ligação completa, sobre os 5 primeiros escores. Na legenda, o tamanho de cada grupo. As cores são só nomes para os grupos.
fig, ax = plt.subplots(
    figsize=(3.8, 3.4))
tams = np.unique(esc)
cores = ["C1", "C3", "C2", "C0"]
marcas = ["D", "^", "s", "o"]
for t, cor, mk in zip(
        tams, cores, marcas):
    m = esc == t
    ax.scatter(escores[m, 0],
        escores[m, 1], s=20,
        marker=mk,
        facecolors="none",
        edgecolors=cor,
        linewidths=1.1,
        label=f"{t} amostras")
ax.set_xlabel("escore na PC1")
ax.set_ylabel("escore na PC2")
ax.legend(ncol=2, fontsize=9,
    loc="lower center",
    bbox_to_anchor=(0.5, 1))
plt.tight_layout()
plt.show()

O grupo de 6, com 4 das amostras de leucemia e as duas K562-repro, é o dos triângulos roxos, juntos numa ponta da nuvem. As outras 2 de leucemia ficam no grupo de 34, o dos círculos azuis.

### Os grupos resistem a perder dados?

Se 10 dos 50 estados fossem deixados de fora, o *k*-means dividiria os outros 40 como na tabela inteira? Para cada $K$ de 2 a 4, o *k*-means roda nos 50 estados padronizados e em 20 subamostras de 40, cada uma sorteada com uma semente. Cada subamostra é padronizada só com os seus 40 estados, como se os outros 10 não existissem. Em cada uma, o índice de Rand ajustado compara a divisão dos 40 com a que eles tinham na tabela inteira:

In [ ]:
Ks_sub = [2, 3, 4]
completo = {}
for K in Ks_sub:
    km_c = KMeans(n_clusters=K,
        n_init=50,
        random_state=15)
    rot = km_c.fit(padr).labels_
    completo[K] = rot
X = prisoes.to_numpy()
aris_sub = {K: [] for K in Ks_sub}
for s in range(20):
    ger_sub = np.random \
        .default_rng(s)
    fica = ger_sub.choice(50, 40,
        replace=False)
    Z = StandardScaler() \
        .fit_transform(X[fica])
    for K in Ks_sub:
        km_s = KMeans(
            n_clusters=K,
            n_init=50,
            random_state=15)
        sub = km_s.fit(Z).labels_
        antes = completo[K][fica]
        aris_sub[K].append(
            adjusted_rand_score(
                antes, sub))
print("K: menor · mediana ·",
    "iguais · <0,5")
for K in Ks_sub:
    v = np.array(aris_sub[K])
    igual = np.isclose(v, 1).sum()
    baixo = (v < 0.5).sum()
    print(f"{K}:",
        numero_br(v.min()), "·",
        numero_br(np.median(v)),
        "·", igual, "·", baixo)

> **🔧 Função**
>
> - **`ger.choice(50, 40, replace=False)`** — sorteia 40 inteiros distintos entre 0 e 49: as posições dos estados que ficam na subamostra. Com `replace=False`, nenhum se repete.
> - **`X[fica]`** e **`completo[K][fica]`** — só as linhas sorteadas, na ordem do sorteio, nos dados e nos rótulos da tabela inteira.
> - **`np.median(v)`** — a mediana dos valores de `v`.

Com $K = 2$, as 20 subamostras dão a mesma divisão da tabela inteira.

Com $K = 3$, só 2 das 20 dão a mesma, e a divisão muda nas outras 18. Com $K = 4$, 5 das 20 dão a mesma, e a divisão muda nas outras 15.

As medianas de $K = 3$ e $K = 4$, 0,88 e 0,92, ficam próximas e altas: na subamostra típica, a divisão muda pouco. A diferença entre os dois está nas piores subamostras: com $K = 3$, 5 das 20 ficam abaixo de 0,5, e o menor índice é 0,38; com $K = 4$, nenhuma, e o menor é 0,65. Com 0,38, tirar 10 estados reorganiza os três grupos. A figura abaixo mostra os 20 valores de cada $K$:

In [ ]:
# Figura: O índice de Rand ajustado entre a divisão do *k*-means em cada uma de 20 subamostras de 40 estados, padronizadas só com os seus 40, e a divisão dos mesmos estados com a tabela inteira, para $K = 2$, $3$ e $4$. Cada ponto é uma subamostra; 1 quer dizer a mesma divisão. Em cada linha, os pontos estão em ordem de valor, espalhados na vertical só para não se esconderem uns atrás dos outros.
fig, ax = plt.subplots(
    figsize=(3.8, 2.6))
desloc = np.linspace(
    -0.3, 0.3, 20)
for K in Ks_sub:
    v = np.sort(aris_sub[K])
    ax.scatter(v, K + desloc,
        s=12, color="C0",
        linewidths=0)
ax.set_yticks(Ks_sub)
ax.set_yticklabels(
    [f"K = {K}" for K in Ks_sub])
ax.set_xlim(0.3, 1.03)
ax.xaxis.set_major_formatter(
    lambda v, _: f"{v:.1f}"
    .replace(".", ","))
ax.set_xlabel(
    "índice de Rand ajustado")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

> **🔧 Função**
>
> - **`np.linspace(-0.3, 0.3, 20)`** — 20 valores igualmente espaçados de −0,3 a 0,3; somados a `K`, põem os 20 pontos de cada linha em alturas diferentes.
> - **`ax.set_yticklabels(lista)`** — troca os números das marcas do eixo vertical pelos textos da lista. **`ax.invert_yaxis()`** põe o menor $K$ em cima.

Nestes dados, a estabilidade não acompanha $K$: a divisão em 2 grupos resistiu às 20 subamostras, e as divisões em 3 e em 4 grupos mudaram na maioria delas, em geral pouco (medianas 0,88 e 0,92), mais a em 3. É uma informação que a inércia não dava, e que ajuda a escolher entre os valores de $K$ que se interpretam bem. Estabilidade é condição necessária para um grupo valer, não prova de que ele existe: uma nuvem contínua, sem grupo nenhum, também pode ser cortada do mesmo jeito em várias subamostras.

### Interpretar com cautela

O *k*-means e o agrupamento hierárquico põem toda observação num grupo, como o ruído mostrou. Quando a maioria das observações forma poucos grupos e umas poucas são muito diferentes de tudo, essas poucas entram mesmo assim em algum grupo e puxam o centroide dele, ou formam grupos próprios.

E uma observação a meio caminho entre dois grupos? Os **modelos de mistura** são uma versão "suave" do *k*-means. Em vez de pôr cada observação num grupo, dão a ela uma probabilidade de pertencer a cada grupo, e essas probabilidades somam 1: uma observação assim recebe, por exemplo, 0,5 e 0,5, onde o *k*-means a poria inteira num só. Cada grupo tem também uma densidade, que diz quão comum é uma observação naquela região. Uma observação atípica tem densidade baixa sob todos os grupos, e isso a deixa à vista em vez de escondida num grupo.

Há também métodos que tentam medir se um grupo encontrado é mais do que o acaso produziria num dado sem grupos, mas não há consenso sobre qual usar. O que se pode fazer sempre é rodar o agrupamento com outras escolhas, de escala, de $K$, de ligação, de semente e de subamostra, e ver o que se repete.

Os grupos continuariam lá com outra decisão? Alguns, sim: a divisão dos estados em 2 grupos sobreviveu às 20 subamostras sorteadas. Outros, não: sem padronizar, os estados se dividiram em faixas de uma coluna só, e em 5 das 20 subamostras a divisão em 3 grupos mudou bastante (índice abaixo de 0,5). E há os que dependem da decisão: o grupo da leucemia apareceu, sobre os genes, com as duas ligações e em 7 das 10 sementes do *k*-means, e se desfez quando o hierárquico rodou sobre os 5 escores. O que se repete sob escolhas diferentes é o que vale relatar, como hipótese a conferir em dados novos.

## Leituras adicionais

- O capítulo 12 de James et al. (2023), com o completamento de matriz (seção 12.3) e o laboratório dele (seção 12.5.2), que usam as componentes principais para preencher os valores ausentes de uma tabela. O PDF gratuito e os dados estão no [site oficial do livro](https://www.statlearning.com).
- As seções 14.3, "Cluster Analysis", e 14.5, "Principal Components, Curves and Surfaces", do capítulo 14 de Hastie et al. (2009), "Unsupervised Learning", num tratamento mais técnico do agrupamento e das componentes principais.
- Os capítulos 8, "Dimensionality Reduction", e 9, "Unsupervised Learning Techniques", de Géron (2022), com a `PCA`, o *k*-means e os modelos de mistura no `scikit-learn`.
- Pearson (1901), o artigo que procura a reta e o plano mais próximos de um sistema de pontos, na origem das componentes principais.
- Hotelling (1933), o artigo que dá às componentes principais o nome e as obtém pela variância, uma de cada vez.
- MacQueen (1967), o artigo que dá ao *k*-means o nome.
- Lloyd (1982), com o algoritmo que alterna médias e reatribuições, o mesmo que o `KMeans` do `scikit-learn` usa por padrão (`algorithm="lloyd"`).
- Ross et al. (2000), o estudo de onde vêm as 64 amostras de linhagens de câncer do `NCI60`.
- Os guias do `scikit-learn` sobre [agrupamento](https://scikit-learn.org/stable/modules/clustering.html) e sobre [componentes principais](https://scikit-learn.org/stable/modules/decomposition.html#pca), com o `KMeans`, o `AgglomerativeClustering` e a `PCA`.

## Referências

- **Géron**. *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*. 3rd ed. O'Reilly Media. 2022.
- **Hastie; Tibshirani; Friedman**. *The Elements of Statistical Learning*. 2nd ed. Springer. 2009.
- **Hotelling**. *Analysis of a Complex of Statistical Variables into Principal Components*. 1933.
- **James; Witten; Hastie; Tibshirani; Taylor**. *An Introduction to Statistical Learning with Applications in Python*. Springer. 2023.
- **Lloyd**. *Least Squares Quantization in {PCM}*. 1982.
- **MacQueen**. *Some Methods for Classification and Analysis of Multivariate Observations*. University of California Press. 1967.
- **Pearson**. *On Lines and Planes of Closest Fit to Systems of Points in Space*. 1901.
- **Ross; Scherf; Eisen; Perou; Rees; Spellman; Iyer; Jeffrey; Van de Rijn; Waltham; Pergamenschikov; Lee; Lashkari; Shalon; Myers; Weinstein; Botstein; Brown**. *Systematic Variation in Gene Expression Patterns in Human Cancer Cell Lines*. 2000.